# Notebook 09: Cross-Model Synthesis and Ideal-TQ Realization Gap

**CPU-only derived analysis.** This notebook reads accepted Llama 3.2-3B and
Qwen2.5-3B evidence, validates it before analysis, and writes only to
`CSE407_cross_model_synthesis_v1/`. It collects no GPU measurements.

Evidence is kept in three explicit tiers: measured systems behavior, the
measured numerically active but uint8/materialized/unfused TQ-Ref realization,
and an ideal perfectly packed/fused analytical counterfactual. Raw joules are
primary; 82 W and 70 W values are trace-clipping sensitivity integrations, not
hardware-cap reruns. Ideal energy remains symbolic because no defensible finite
domain for effective joules per modeled byte or non-traffic overhead is
registered.


## Run 1 - Imports, paths, and immutable-source contract

Set `CSE407_PROJECT_ROOT` only when the notebook is not launched from the
project root. The source notebook itself is normalized and hashed so outputs and
execution counters cannot alter the analysis-source identity.


In [ ]:
import ast
import hashlib
import json
import math
import os
import platform
import re
import statistics
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCHEMA = "cse407-cross-model-synthesis-v1-2026-08-05-1"
OUTPUT_NAME = "CSE407_cross_model_synthesis_v1"
BOOTSTRAP_REPS = 2000
BOOTSTRAP_SEED = 909407
FP_BYTES = 2
NORM_BYTES = 4
RESIDUAL_TOKENS = 128
DECODE_TOKENS = 64
ALIGNMENT_BYTES = 1
METADATA_BYTES_PER_LAYER = 0

def locate_project_root():
    candidates = []
    if os.environ.get("CSE407_PROJECT_ROOT"):
        candidates.append(Path(os.environ["CSE407_PROJECT_ROOT"]))
    candidates.extend([
        Path.cwd(),
        Path("/content/drive/MyDrive/CSE407/Project"),
        Path("/content/drive/MyDrive/Project"),
    ])
    required = {"AGENTS.md", "research", "experiments", "CSE407_llama_v5_final"}
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if all((candidate / item).exists() for item in required):
            return candidate
    raise FileNotFoundError("Set CSE407_PROJECT_ROOT to the project root")

PROJECT_ROOT = locate_project_root()
RESEARCH = PROJECT_ROOT / "research"
NOTEBOOK_PATH = PROJECT_ROOT / "experiments/track_a_failsafe_colab/09_cross_model_synthesis_v1.ipynb"
LLAMA_V5 = PROJECT_ROOT / "CSE407_llama_v5_final"
LLAMA_HARD = PROJECT_ROOT / "CSE407_llama_track_a_full_v1"
LLAMA_POWER = PROJECT_ROOT / "CSE407_llama_power_robustness_v1"
QWEN = PROJECT_ROOT / "CSE407_qwen_track_a_reference_v1"
OUTPUT_ROOT = PROJECT_ROOT / OUTPUT_NAME

IMMUTABLE_ROOTS = [LLAMA_V5, LLAMA_HARD, LLAMA_POWER, QWEN]
for root in IMMUTABLE_ROOTS:
    assert root.is_dir(), root
    assert root.name.startswith("CSE407_")
assert not OUTPUT_ROOT.exists(), f"Refusing to merge or overwrite {OUTPUT_ROOT}"
assert NOTEBOOK_PATH.is_file(), NOTEBOOK_PATH
assert "torch" not in globals()

EXPECTED = {
    "llama_v5": {
        "root": LLAMA_V5, "schema": "v5-final-2026-07-31-1",
        "fingerprint": "8d27af8c9ee94de3", "session": "20260731T125351-ad1b5249",
        "manifest": RESEARCH / "llama_v5_artifact_manifest.json",
        "manifest_sha256": "a7087d7e2e24a929df0f62177bc6fc1e7be38cd1bf12d8ffaf9849f6c7677b19",
    },
    "llama_hard": {
        "root": LLAMA_HARD, "schema": "track-a-llama-full-v1-2026-08-03-1",
        "fingerprint": "106a561c07c62937", "session": "20260803T153710-2a4dc573",
        "manifest": RESEARCH / "llama_track_a_full_v1_artifact_manifest.json",
        "manifest_sha256": "2a4c996f32916b36aa11ad3e18104e6378ed7ad7a54230d8f728b7c1f52ba9bc",
    },
    "llama_power": {
        "root": LLAMA_POWER, "schema": "track-a-llama-power-robustness-v1-2026-08-03-1",
        "fingerprint": "91ebc63773e735e6",
        "manifest": RESEARCH / "llama_power_robustness_v1_artifact_manifest.json",
        "manifest_sha256": "b0e38005ea576a744122161ba671213936bf624de136bdf0b263cc0347ce1071",
    },
    "qwen": {
        "root": QWEN, "schema": "track-a-qwen-full-reference-v1-2026-08-04-1",
        "fingerprint": "861d89617e53034d", "session": "20260804T154752-97822cdb",
        "manifest": QWEN / "artifact_manifest.json",
        "manifest_sha256": "975cb8f19405f9d7e4267bc4a8143ead943c853776c6ba88b8697695ac78c1c9",
    },
}

print({"project_root": str(PROJECT_ROOT), "cpu_only": True, "schema": SCHEMA})


## Run 2 - Deterministic hashing, safe I/O, and source binding

Every output write is containment-checked. Accepted roots are read-only. The
normalized notebook hash includes cell IDs, types, and source text while
excluding outputs and execution state.


In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_json_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=True, allow_nan=False).encode("utf-8")

def canonical_sha256(value):
    return hashlib.sha256(canonical_json_bytes(value)).hexdigest()

def no_duplicate_object(pairs):
    out = {}
    for key, value in pairs:
        if key in out:
            raise ValueError(f"Duplicate JSON key: {key}")
        out[key] = value
    return out

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        value = json.load(handle, object_pairs_hook=no_duplicate_object,
                          parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))
    return value

def normalized_notebook_hash(path):
    notebook = read_json(path)
    payload = {
        "nbformat": notebook["nbformat"],
        "nbformat_minor": notebook["nbformat_minor"],
        "cells": [{
            "id": cell.get("id"), "cell_type": cell["cell_type"],
            "source": "".join(cell.get("source", [])),
        } for cell in notebook["cells"]],
    }
    return canonical_sha256(payload)

NOTEBOOK_SOURCE_SHA256 = normalized_notebook_hash(NOTEBOOK_PATH)

def output_path(name):
    path = (OUTPUT_ROOT / name).resolve()
    assert path == OUTPUT_ROOT.resolve() or OUTPUT_ROOT.resolve() in path.parents
    assert all(root.resolve() not in path.parents and path != root.resolve()
               for root in IMMUTABLE_ROOTS)
    return path

def atomic_write_bytes(path, payload):
    path = Path(path).resolve()
    assert OUTPUT_ROOT.resolve() in path.parents
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

def write_json(name, value):
    payload = json.dumps(value, sort_keys=True, indent=2, ensure_ascii=True,
                         allow_nan=False).encode("utf-8") + b"\n"
    atomic_write_bytes(output_path(name), payload)

def write_csv(name, frame):
    text = frame.to_csv(index=False, lineterminator="\n", float_format="%.17g")
    atomic_write_bytes(output_path(name), text.encode("utf-8"))

def write_text(name, text):
    atomic_write_bytes(output_path(name), text.rstrip().encode("utf-8") + b"\n")

def bool_value(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    if value in (0, 1):
        return bool(value)
    raise ValueError(f"Not a strict boolean: {value!r}")

print("Notebook source SHA-256:", NOTEBOOK_SOURCE_SHA256)


## Run 3 - Fail-closed source manifests and acceptance markers

All listed files are checked by path, byte size, and SHA-256 before measurement
rows are loaded. Llama v5 may omit only its non-scientific `.gitignore` when
copied from Drive. Qwen's own manifest and final-integrity marker are the two
declared files outside its 591-entry manifest.


In [ ]:
SOURCE_GATES = []
INPUT_FILES = []

def gate(source_id, gate_id, expected, observed, evidence_path, note=""):
    passed = expected == observed
    SOURCE_GATES.append({
        "source_id": source_id, "gate_id": gate_id,
        "expected": expected, "observed": observed, "passed": passed,
        "evidence_path": str(Path(evidence_path).relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "evidence_sha256": sha256_file(evidence_path), "note": note,
    })
    assert passed, SOURCE_GATES[-1]

def verify_manifest(source_id, allowed_missing=frozenset(), allowed_unlisted=frozenset()):
    spec = EXPECTED[source_id]
    manifest_path = spec["manifest"]
    gate(source_id, "manifest_sha256", spec["manifest_sha256"],
         sha256_file(manifest_path), manifest_path)
    manifest = read_json(manifest_path)
    entries = manifest["files"]
    root = spec["root"].resolve()
    listed = set()
    for entry in entries:
        relative = Path(entry["path"])
        assert not relative.is_absolute() and ".." not in relative.parts
        path = (root / relative).resolve()
        assert root in path.parents
        listed.add(relative.as_posix())
        if not path.exists() and relative.as_posix() in allowed_missing:
            continue
        assert path.is_file() and not path.is_symlink(), path
        assert path.stat().st_size == int(entry["bytes"]), path
        assert sha256_file(path) == entry["sha256"], path
        INPUT_FILES.append({
            "source_id": source_id,
            "path": str(path.relative_to(PROJECT_ROOT)).replace("\\", "/"),
            "bytes": path.stat().st_size, "sha256": entry["sha256"],
            "role": "manifest_verified_artifact", "canonical": path.suffix in {".json", ".csv"},
        })
    observed = {
        str(path.relative_to(root)).replace("\\", "/")
        for path in root.rglob("*") if path.is_file()
    }
    missing = listed - observed - set(allowed_missing)
    unexpected = observed - listed - set(allowed_unlisted)
    assert not missing, sorted(missing)[:10]
    assert not unexpected, sorted(unexpected)[:10]
    INPUT_FILES.append({
        "source_id": source_id,
        "path": str(manifest_path.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "bytes": manifest_path.stat().st_size, "sha256": sha256_file(manifest_path),
        "role": "external_immutable_manifest", "canonical": True,
    })
    return manifest

v5_manifest = verify_manifest("llama_v5", allowed_missing=frozenset({".gitignore"}))
hard_manifest = verify_manifest("llama_hard")
power_manifest = verify_manifest("llama_power")
qwen_manifest = verify_manifest(
    "qwen",
    allowed_unlisted=frozenset({"artifact_manifest.json", "analysis/final_integrity.json"}),
)

v5_environment = read_json(LLAMA_V5 / "v5_final_environment.json")
hard_integrity = read_json(LLAMA_HARD / "analysis/final_integrity.json")
power_integrity = read_json(LLAMA_POWER / "final_integrity.json")
qwen_integrity = read_json(QWEN / "analysis/final_integrity.json")

gate("llama_v5", "schema", EXPECTED["llama_v5"]["schema"],
     v5_environment["schema_version"], LLAMA_V5 / "v5_final_environment.json")
gate("llama_v5", "fingerprint", EXPECTED["llama_v5"]["fingerprint"],
     v5_manifest["facts"]["fingerprint"], EXPECTED["llama_v5"]["manifest"])
gate("llama_v5", "primary_rows", 380, v5_manifest["facts"]["primary_execution_rows"],
     EXPECTED["llama_v5"]["manifest"])
gate("llama_v5", "traces", 95, v5_manifest["facts"]["power_trace_count"],
     EXPECTED["llama_v5"]["manifest"])

for key, expected in [("passed", True), ("schema_version", EXPECTED["llama_hard"]["schema"]),
                      ("fingerprint", EXPECTED["llama_hard"]["fingerprint"]),
                      ("quality_rows", 300), ("energy_rows", 5)]:
    gate("llama_hard", key, expected, hard_integrity[key], LLAMA_HARD / "analysis/final_integrity.json")

for key, expected in [("passed", True), ("schema_version", EXPECTED["llama_power"]["schema"]),
                      ("fingerprint", EXPECTED["llama_power"]["fingerprint"]),
                      ("trace_count", 100), ("source_manifests_verified", True)]:
    gate("llama_power", key, expected, power_integrity[key], LLAMA_POWER / "final_integrity.json")

qwen_expected = {
    "passed": True, "schema_version": EXPECTED["qwen"]["schema"],
    "fingerprint": EXPECTED["qwen"]["fingerprint"],
    "session_id": EXPECTED["qwen"]["session"], "core_energy_rows": 75,
    "sensitivity_energy_rows": 20, "power_traces": 95,
    "core_quality_rows": 300, "sensitivity_quality_rows": 60,
    "unique_measurement_keys": 455, "q1s_measurement_rows": 0,
    "artifact_manifest_sha256": EXPECTED["qwen"]["manifest_sha256"],
    "semantic_sha256": "fffa82607af6e7d2c892d18d5bc790524570833472ad397b7ab2e54eb03005ef",
    "prompt_set_sha256": "115c1568fea7aeea038a46ce309d0c8bcaf1f5405544466ad6f379ee7e94ae3a",
}
for key, expected in qwen_expected.items():
    gate("qwen", key, expected, qwen_integrity[key], QWEN / "analysis/final_integrity.json")

for path, source_id, role in [
    (LLAMA_HARD / "analysis/final_integrity.json", "llama_hard", "acceptance_marker"),
    (LLAMA_POWER / "final_integrity.json", "llama_power", "acceptance_marker"),
    (QWEN / "artifact_manifest.json", "qwen", "artifact_manifest"),
    (QWEN / "analysis/final_integrity.json", "qwen", "acceptance_marker"),
    (QWEN / "analysis/analysis_provenance.json", "qwen", "analysis_provenance"),
]:
    INPUT_FILES.append({"source_id": source_id,
        "path": str(path.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "bytes": path.stat().st_size, "sha256": sha256_file(path),
        "role": role, "canonical": True})

assert all(row["passed"] for row in SOURCE_GATES)
print(f"Validated {len(INPUT_FILES)} manifest-bound input files and {len(SOURCE_GATES)} gates")


## Run 4 - Canonical shards, CSV correspondence, and raw traces

JSON shards are canonical where available. Historical Llama v5 has no row-level
JSON shards, so its externally checksummed CSV snapshots are the canonical rows.
All trace phases are integrated separately because phase timestamps reset.


In [ ]:
def equivalent(json_value, csv_value):
    if not isinstance(json_value, (list, dict)) and pd.isna(json_value):
        return pd.isna(csv_value)
    if json_value is None:
        return pd.isna(csv_value)
    if isinstance(json_value, bool):
        try:
            return bool_value(csv_value) == json_value
        except ValueError:
            return False
    if isinstance(json_value, (list, dict)):
        if isinstance(csv_value, str):
            try:
                parsed = ast.literal_eval(csv_value)
            except Exception:
                try:
                    parsed = json.loads(csv_value)
                except Exception:
                    return False
        else:
            parsed = csv_value
        return canonical_json_bytes(parsed) == canonical_json_bytes(json_value)
    if isinstance(json_value, (int, float)) and not isinstance(json_value, bool):
        try:
            return math.isclose(float(json_value), float(csv_value), rel_tol=1e-12, abs_tol=1e-12)
        except Exception:
            return False
    return str(csv_value) == str(json_value)

def load_shards(directory, expected_count, identity, key_columns):
    paths = sorted(Path(directory).glob("*.json"))
    assert len(paths) == expected_count, (directory, len(paths))
    rows = [read_json(path) for path in paths]
    for row in rows:
        for key, expected in identity.items():
            assert row[key] == expected, (key, row.get(key), expected)
        if "structural_pass" in row:
            assert row["structural_pass"] is True
    frame = pd.DataFrame(rows)
    assert not frame.duplicated(list(key_columns)).any()
    return frame

def assert_csv_correspondence(shards, csv_path, key_columns):
    snapshot = pd.read_csv(csv_path)
    assert len(snapshot) == len(shards)
    assert set(snapshot.columns) == set(shards.columns), (csv_path, set(snapshot.columns) ^ set(shards.columns))
    left = {tuple(row[col] for col in key_columns): row for row in shards.to_dict("records")}
    for csv_row in snapshot.to_dict("records"):
        key = tuple(csv_row[col] for col in key_columns)
        assert key in left, (csv_path, key)
        json_row = left[key]
        for column in snapshot.columns:
            assert equivalent(json_row[column], csv_row[column]), (csv_path, key, column)

v5_core_energy = pd.read_csv(LLAMA_V5 / "v5_final_energy_results.csv")
v5_sens_energy = pd.read_csv(LLAMA_V5 / "v5_final_sensitivity_energy_4096.csv")
v5_core_quality = pd.read_csv(LLAMA_V5 / "v5_final_quality_results.csv")
v5_sens_quality = pd.read_csv(LLAMA_V5 / "v5_final_sensitivity_quality_4096.csv")
assert [len(x) for x in [v5_core_energy, v5_sens_energy, v5_core_quality, v5_sens_quality]] == [75, 20, 225, 60]
for frame in [v5_core_energy, v5_sens_energy, v5_core_quality, v5_sens_quality]:
    assert set(frame["fingerprint"]) == {EXPECTED["llama_v5"]["fingerprint"]}
    assert frame["structural_pass"].map(bool_value).all()
assert not v5_core_energy.duplicated(["variant", "context", "block"]).any()
assert not v5_sens_energy.duplicated(["variant", "context", "block"]).any()
assert not v5_core_quality.duplicated(["variant", "context", "depth", "seed"]).any()
assert not v5_sens_quality.duplicated(["variant", "context", "depth", "seed"]).any()
assert set(map(tuple, v5_core_energy[["variant", "context"]].drop_duplicates().to_numpy())) == {
    (variant, context) for variant in ["C0", "C1", "C2", "C3", "C4"] for context in [1024, 2048, 4096]
}
gate("llama_v5", "core_energy_unique_keys", 75,
     len(v5_core_energy.drop_duplicates(["variant", "context", "block"])), LLAMA_V5 / "v5_final_energy_results.csv")
gate("llama_v5", "sensitivity_energy_unique_keys", 20,
     len(v5_sens_energy.drop_duplicates(["variant", "context", "block"])), LLAMA_V5 / "v5_final_sensitivity_energy_4096.csv")
gate("llama_v5", "core_quality_unique_keys", 225,
     len(v5_core_quality.drop_duplicates(["variant", "context", "depth", "seed"])), LLAMA_V5 / "v5_final_quality_results.csv")
gate("llama_v5", "sensitivity_quality_unique_keys", 60,
     len(v5_sens_quality.drop_duplicates(["variant", "context", "depth", "seed"])), LLAMA_V5 / "v5_final_sensitivity_quality_4096.csv")

hard_energy = load_shards(LLAMA_HARD / "energy_rows", 5,
    {"fingerprint": EXPECTED["llama_hard"]["fingerprint"], "schema_version": EXPECTED["llama_hard"]["schema"]},
    ["variant", "context", "block"])
hard_quality = load_shards(LLAMA_HARD / "quality_rows", 300,
    {"fingerprint": EXPECTED["llama_hard"]["fingerprint"], "schema_version": EXPECTED["llama_hard"]["schema"]},
    ["variant", "scenario_id"])
assert_csv_correspondence(hard_energy, LLAMA_HARD / "track_a_c1w_energy_control.csv", ["variant", "context", "block"])
assert_csv_correspondence(hard_quality, LLAMA_HARD / "track_a_full_quality_results.csv", ["variant", "scenario_id"])

qwen_core_energy = load_shards(QWEN / "core_energy_rows", 75,
    {"fingerprint": EXPECTED["qwen"]["fingerprint"], "schema_version": EXPECTED["qwen"]["schema"], "table": "core"},
    ["table", "variant", "context", "block"])
qwen_sens_energy = load_shards(QWEN / "sensitivity_energy_rows", 20,
    {"fingerprint": EXPECTED["qwen"]["fingerprint"], "schema_version": EXPECTED["qwen"]["schema"], "table": "sensitivity"},
    ["table", "variant", "context", "block"])
qwen_core_quality = load_shards(QWEN / "core_quality_rows", 300,
    {"fingerprint": EXPECTED["qwen"]["fingerprint"], "schema_version": EXPECTED["qwen"]["schema"], "table": "core"},
    ["table", "variant", "scenario_id"])
qwen_sens_quality = load_shards(QWEN / "sensitivity_quality_rows", 60,
    {"fingerprint": EXPECTED["qwen"]["fingerprint"], "schema_version": EXPECTED["qwen"]["schema"], "table": "sensitivity"},
    ["table", "variant", "scenario_id"])
assert_csv_correspondence(qwen_core_energy, QWEN / "core_energy_results.csv", ["table", "variant", "context", "block"])
assert_csv_correspondence(qwen_sens_energy, QWEN / "sensitivity_energy_results.csv", ["table", "variant", "context", "block"])
assert_csv_correspondence(qwen_core_quality, QWEN / "core_quality_results.csv", ["table", "variant", "scenario_id"])
assert_csv_correspondence(qwen_sens_quality, QWEN / "sensitivity_quality_results.csv", ["table", "variant", "scenario_id"])

def integrate_phase(samples, cap=None):
    assert len(samples) >= 2
    elapsed = np.asarray([float(x["elapsed_s"]) for x in samples], dtype=float)
    power = np.asarray([float(x["power_w"]) for x in samples], dtype=float)
    assert np.isfinite(elapsed).all() and np.isfinite(power).all()
    assert (np.diff(elapsed) > 0).all() and (power >= 0).all()
    if cap is not None:
        power = np.minimum(power, float(cap))
    return float(np.trapezoid(power, elapsed))

def trace_integrals(path):
    trace = read_json(path)
    out = {}
    for regime, cap in [("raw", None), ("cap82", 82.0), ("cap70", 70.0)]:
        out[f"prefill_{regime}"] = integrate_phase(trace["prefill"], cap)
        out[f"decode_{regime}"] = integrate_phase(trace["decode"], cap)
        out[f"total_{regime}"] = out[f"prefill_{regime}"] + out[f"decode_{regime}"]
    return out

llama_trace_table = pd.read_csv(LLAMA_POWER / "trace_level_power_robustness.csv")
assert len(llama_trace_table) == 100
for row in llama_trace_table.to_dict("records"):
    root = LLAMA_V5 if row["source"] == "v5" else LLAMA_HARD
    path = root / "power_traces" / row["trace_name"]
    values = trace_integrals(path)
    for regime in ["raw", "cap82", "cap70"]:
        for phase in ["prefill", "decode", "total"]:
            assert math.isclose(values[f"{phase}_{regime}"], row[f"{phase}_energy_{regime}_j"], abs_tol=1e-9)

for row in pd.concat([qwen_core_energy, qwen_sens_energy], ignore_index=True).to_dict("records"):
    path = QWEN / "power_traces" / row["trace_name"]
    assert sha256_file(path) == row["trace_sha256"]
    values = trace_integrals(path)
    for regime, prefix in [("raw", ""), ("cap82", "cap82_"), ("cap70", "cap70_")]:
        for phase in ["prefill", "decode", "total"]:
            assert math.isclose(values[f"{phase}_{regime}"], row[f"{prefix}{phase}_energy_j"], abs_tol=1e-9)

assert len(list((LLAMA_V5 / "power_traces").glob("*.json"))) == 95
assert len(list((LLAMA_HARD / "power_traces").glob("*.json"))) == 5
assert len(list((QWEN / "power_traces").glob("*.json"))) == 95
print("Canonical rows and 195 raw traces passed")


## Run 5 - Normalized empirical tables

Source labels are retained. `normalized_role` is derived only for cross-model
synthesis. Historical Llama uses stock C1 across contexts; Qwen uses Q1-W. The
separate Llama C1-W control exists only at 4096 and remains cross-session
descriptive.


In [ ]:
ROLE = {
    "C0": "fp16", "C1": "awq_stock", "C1-W": "awq_wrapper",
    "C2": "awq_eviction", "C3": "awq_tq_ref", "C3-Ref": "awq_tq_ref",
    "C4": "awq_eviction_tq_ref", "C4-Ref": "awq_eviction_tq_ref",
    "Q0": "fp16", "Q1-W": "awq_wrapper", "Q2": "awq_eviction",
    "Q3-Ref": "awq_tq_ref", "Q4-Ref": "awq_eviction_tq_ref",
    "C3-TQ2": "awq_tq_ref", "C3-TQ3": "awq_tq_ref",
    "Q3-Ref-TQ2": "awq_tq_ref", "Q3-Ref-TQ3": "awq_tq_ref",
}

llama_power_lookup = llama_trace_table.set_index(["variant_original", "context", "block"])

def normalize_energy(frame, model, experiment, table, session, fingerprint, schema, power_lookup=None):
    rows = []
    for source in frame.to_dict("records"):
        variant = source["variant"]
        trace_name = source.get("trace_name") or Path(str(source.get("trace_file", ""))).name
        trace_hash = source.get("trace_sha256")
        if not trace_hash:
            trace_root = LLAMA_V5 if experiment == "llama_v5" else LLAMA_HARD
            trace_hash = sha256_file(trace_root / "power_traces" / trace_name)
        if power_lookup is not None:
            power_row = power_lookup.loc[(variant, int(source["context"]), int(source["block"]))]
        for regime, prefix in [("raw", ""), ("cap82", "cap82_"), ("cap70", "cap70_")]:
            if regime == "raw":
                prefill, decode, total = (source["prefill_energy_j"], source["decode_energy_j"], source["total_energy_j"])
            elif power_lookup is not None:
                prefill, decode, total = (power_row[f"prefill_energy_{regime}_j"], power_row[f"decode_energy_{regime}_j"], power_row[f"total_energy_{regime}_j"])
            else:
                prefill, decode, total = (source[f"{prefix}prefill_energy_j"], source[f"{prefix}decode_energy_j"], source[f"{prefix}total_energy_j"])
            rows.append({
                "model": model, "source_experiment": experiment, "source_session": session,
                "source_schema": schema, "source_fingerprint": fingerprint,
                "source_table": table, "source_variant": variant,
                "normalized_role": ROLE.get(variant, variant), "context": int(source["context"]),
                "block": int(source["block"]), "average_budget": int(source.get("average_budget", 1024)),
                "tq_bits": int(source.get("tq_bits", 4)), "regime": regime,
                "prefill_energy_j": float(prefill), "decode_energy_j": float(decode),
                "total_energy_j": float(total), "prefill_s": float(source["prefill_s"]),
                "decode_s": float(source["decode_s"]), "decode_tok_s": float(source["decode_tok_s"]),
                "post_prefill_cache_live_bytes": int(source.get("post_prefill_cache_live_bytes", 0)),
                "post_decode_cache_live_bytes": int(source.get("post_decode_cache_live_bytes", 0)),
                "post_prefill_retained_fp16_bytes": int(source.get("post_prefill_retained_fp16_bytes", source.get("post_prefill_cache_live_bytes", 0))),
                "post_decode_retained_fp16_bytes": int(source.get("post_decode_retained_fp16_bytes", source.get("post_decode_cache_live_bytes", 0))),
                "post_prefill_tq_index_bytes": int(source.get("post_prefill_tq_index_bytes", 0)),
                "post_decode_tq_index_bytes": int(source.get("post_decode_tq_index_bytes", 0)),
                "post_prefill_tq_norm_bytes": int(source.get("post_prefill_tq_norm_bytes", 0)),
                "post_decode_tq_norm_bytes": int(source.get("post_decode_tq_norm_bytes", 0)),
                "post_prefill_tq_materialized_bytes": int(source.get("post_prefill_tq_materialized_bytes", 0)),
                "post_decode_tq_materialized_bytes": int(source.get("post_decode_tq_materialized_bytes", 0)),
                "post_prefill_tq_package_reported_bytes": int(source.get("post_prefill_tq_package_reported_bytes", 0)),
                "post_decode_tq_package_reported_bytes": int(source.get("post_decode_tq_package_reported_bytes", 0)),
                "structural_pass": bool_value(source["structural_pass"]), "trace_name": trace_name,
                "trace_sha256": trace_hash,
                "baseline_policy": "stock_C1_historical" if model == "llama" and experiment == "llama_v5" else ("separate_session_C1W_control" if experiment == "llama_hard" else "matched_wrapper_Q1W"),
            })
    return pd.DataFrame(rows)

energy = pd.concat([
    normalize_energy(v5_core_energy, "llama", "llama_v5", "core", EXPECTED["llama_v5"]["session"], EXPECTED["llama_v5"]["fingerprint"], EXPECTED["llama_v5"]["schema"], llama_power_lookup),
    normalize_energy(v5_sens_energy, "llama", "llama_v5", "sensitivity", EXPECTED["llama_v5"]["session"], EXPECTED["llama_v5"]["fingerprint"], EXPECTED["llama_v5"]["schema"], llama_power_lookup),
    normalize_energy(hard_energy, "llama", "llama_hard", "control", EXPECTED["llama_hard"]["session"], EXPECTED["llama_hard"]["fingerprint"], EXPECTED["llama_hard"]["schema"], llama_power_lookup),
    normalize_energy(qwen_core_energy, "qwen", "qwen_full", "core", EXPECTED["qwen"]["session"], EXPECTED["qwen"]["fingerprint"], EXPECTED["qwen"]["schema"]),
    normalize_energy(qwen_sens_energy, "qwen", "qwen_full", "sensitivity", EXPECTED["qwen"]["session"], EXPECTED["qwen"]["fingerprint"], EXPECTED["qwen"]["schema"]),
], ignore_index=True)

def normalize_quality(frame, model, experiment, table, session, fingerprint, schema, population):
    rows = []
    for source in frame.to_dict("records"):
        variant = source["variant"]
        rows.append({
            "model": model, "source_experiment": experiment, "source_session": session,
            "source_schema": schema, "source_fingerprint": fingerprint,
            "source_table": table, "source_variant": variant,
            "normalized_role": ROLE.get(variant, variant), "quality_population": population,
            "scenario_id": str(source.get("scenario_id", f"d{source.get('depth')}-s{source.get('seed')}")),
            "depth": float(source["depth"]), "context": int(source["context"]),
            "average_budget": int(source.get("average_budget", 1024)),
            "tq_bits": int(source.get("tq_bits", 4)),
            "input_ids_sha256": source.get("input_ids_sha256", "historical_not_persisted"),
            "retrieval_correct": bool_value(source["retrieval_correct"]),
            "format_correct": bool_value(source["format_correct"]),
            "legacy_target_anywhere": bool_value(source.get("legacy_target_anywhere", source["retrieval_correct"])),
            "failure_category": source.get("failure_category", "historical_binary"),
            "structural_pass": bool_value(source["structural_pass"]),
        })
    return pd.DataFrame(rows)

quality = pd.concat([
    normalize_quality(hard_quality, "llama", "llama_hard", "core", EXPECTED["llama_hard"]["session"], EXPECTED["llama_hard"]["fingerprint"], EXPECTED["llama_hard"]["schema"], "hard60"),
    normalize_quality(qwen_core_quality, "qwen", "qwen_full", "core", EXPECTED["qwen"]["session"], EXPECTED["qwen"]["fingerprint"], EXPECTED["qwen"]["schema"], "hard60"),
    normalize_quality(qwen_sens_quality, "qwen", "qwen_full", "sensitivity", EXPECTED["qwen"]["session"], EXPECTED["qwen"]["fingerprint"], EXPECTED["qwen"]["schema"], "qwen_common15"),
    normalize_quality(v5_core_quality, "llama", "llama_v5", "historical_core", EXPECTED["llama_v5"]["session"], EXPECTED["llama_v5"]["fingerprint"], EXPECTED["llama_v5"]["schema"], "historical_easy"),
], ignore_index=True)

assert len(energy) == (75 + 20 + 5 + 75 + 20) * 3
assert len(quality) == 300 + 300 + 60 + 225
assert energy["structural_pass"].all() and quality["structural_pass"].all()
print({"normalized_energy_rows": len(energy), "normalized_quality_rows": len(quality)})


## Run 6 - Empirical effects, interactions, Wilson intervals, and Pareto stability

Cross-model absolute joules are never paired. Within-session block pairing is
retained for empirical bootstrap samples; quality scenarios are resampled jointly
within depth. Energy and quality are resampled independently. Frontier frequency
means resampling selection stability, not probability of the true frontier.


In [ ]:
def wilson(successes, n, z=1.959963984540054):
    assert n > 0
    p = successes / n
    denominator = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denominator
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
    return max(0.0, center - half), min(1.0, center + half)

quality_rows = []
hard = quality.query("quality_population == 'hard60'")
for keys, group in hard.groupby(["model", "quality_population", "source_variant", "tq_bits", "depth"], dropna=False):
    successes = int(group["retrieval_correct"].sum())
    low, high = wilson(successes, len(group))
    quality_rows.append(dict(zip(["model", "quality_population", "source_variant", "tq_bits", "depth"], keys),
        successes=successes, n=len(group), estimate=successes/len(group), wilson95_low=low, wilson95_high=high))
for keys, group in hard.groupby(["model", "quality_population", "source_variant", "tq_bits"], dropna=False):
    successes = int(group["retrieval_correct"].sum())
    low, high = wilson(successes, len(group))
    quality_rows.append(dict(zip(["model", "quality_population", "source_variant", "tq_bits"], keys),
        depth="all", successes=successes, n=len(group), estimate=successes/len(group), wilson95_low=low, wilson95_high=high))

common_ids = set(qwen_sens_quality.query("variant == 'Q3-Ref-TQ2'")["scenario_id"])
assert len(common_ids) == 15
qwen_common = pd.concat([
    quality.query("model == 'qwen' and quality_population == 'qwen_common15' and source_variant in ['Q3-Ref-TQ2','Q3-Ref-TQ3']"),
    quality.query("model == 'qwen' and quality_population == 'hard60' and source_variant == 'Q3-Ref' and scenario_id in @common_ids").assign(quality_population="qwen_common15"),
], ignore_index=True)
for keys, group in qwen_common.groupby(["model", "quality_population", "source_variant", "tq_bits"], dropna=False):
    successes = int(group["retrieval_correct"].sum())
    low, high = wilson(successes, len(group))
    quality_rows.append(dict(zip(["model", "quality_population", "source_variant", "tq_bits"], keys),
        depth="all", successes=successes, n=len(group), estimate=successes/len(group), wilson95_low=low, wilson95_high=high))
quality_intervals = pd.DataFrame(quality_rows)

effect_rows = []
rng_effect = np.random.default_rng(BOOTSTRAP_SEED + 17)
comparisons = {
    "llama": [("C1", "C2", "paired_within_v5_wrapper_confounded"), ("C1", "C3", "paired_within_v5_wrapper_confounded"),
              ("C1-W", "C2", "separate_session_descriptive_4096_only")],
    "qwen": [("Q1-W", "Q2", "paired_within_session"), ("Q1-W", "Q3-Ref", "paired_within_session")],
}
for model, pairs in comparisons.items():
    for baseline, comparison, design in pairs:
        contexts = [4096] if baseline == "C1-W" else [1024, 2048, 4096]
        for context in contexts:
            for regime in ["raw", "cap82", "cap70"]:
                for metric in ["total_energy_j", "decode_energy_j", "decode_tok_s"]:
                    base = energy.query("model == @model and source_variant == @baseline and context == @context and regime == @regime")[metric]
                    comp = energy.query("model == @model and source_variant == @comparison and context == @context and regime == @regime")[metric]
                    if len(base) != 5 or len(comp) != 5:
                        continue
                    b, c = float(base.mean()), float(comp.mean())
                    samples = []
                    for _ in range(BOOTSTRAP_REPS):
                        if design.startswith("paired_within"):
                            idx = rng_effect.integers(0, 5, 5)
                            samples.append(float(comp.to_numpy()[idx].mean() - base.to_numpy()[idx].mean()))
                        else:
                            ib = rng_effect.integers(0, 5, 5)
                            ic = rng_effect.integers(0, 5, 5)
                            samples.append(float(comp.to_numpy()[ic].mean() - base.to_numpy()[ib].mean()))
                    ci_low, ci_high = np.quantile(samples, [0.025, 0.975])
                    effect_rows.append({"model": model, "context": context, "regime": regime, "metric": metric,
                        "baseline_variant": baseline, "comparison_variant": comparison,
                        "n_baseline": len(base), "n_comparison": len(comp), "mean_baseline": b,
                        "mean_comparison": c, "absolute_change": c-b,
                        "relative_change": (c-b)/b if b else np.nan,
                        "bootstrap_ci95_low": ci_low, "bootstrap_ci95_high": ci_high,
                        "p_adjusted": np.nan, "p_adjusted_source": "see registered_significance_results.csv",
                        "design": design,
                        "inference_allowed": design == "paired_within_session",
                        "interpretation": "within-model; cross-model causation not inferred"})
within_effects = pd.DataFrame(effect_rows)

interaction_rows = []
llama_registered = pd.read_csv(LLAMA_V5 / "v5_final_interaction_tests.csv")
for row in llama_registered.to_dict("records"):
    interaction_rows.append({"model": "llama", "regime": "raw", "metric": row["metric"],
        "n": 5, "mean_interaction": row["mean_difference_in_differences"],
        "ci95_low": row["ci95_low"], "ci95_high": row["ci95_high"],
        "p_raw": row["p_unadjusted"], "p_adjusted": row["p_holm"],
        "source": "registered_v5", "baseline_wrapper_matched": False,
        "caveat": "configuration-level interaction; stock C1 versus wrapped C2-C4"})
for regime in ["cap82", "cap70"]:
    subset = energy.query("model == 'llama' and source_experiment == 'llama_v5' and context == 4096 and regime == @regime")
    for metric in ["total_energy_j", "decode_energy_j"]:
        pivot = subset.pivot(index="block", columns="source_variant", values=metric)
        values = (pivot["C4"] - pivot["C3"]) - (pivot["C2"] - pivot["C1"])
        interaction_rows.append({"model": "llama", "regime": regime, "metric": metric,
            "n": 5, "mean_interaction": values.mean(), "ci95_low": np.nan, "ci95_high": np.nan,
            "p_raw": np.nan, "p_adjusted": np.nan, "source": "derived_capped_descriptive",
            "baseline_wrapper_matched": False, "caveat": "trace-clipping sensitivity; no new inferential decision"})
qwen_interactions = pd.read_csv(QWEN / "analysis/interaction_tests_by_regime.csv")
for row in qwen_interactions.to_dict("records"):
    interaction_rows.append({"model": "qwen", "regime": row["regime"], "metric": row["metric"],
        "n": row["n"], "mean_interaction": row["mean_interaction"], "ci95_low": np.nan, "ci95_high": np.nan,
        "p_raw": row["p_raw"], "p_adjusted": row["p_holm_4"], "source": "registered_qwen",
        "baseline_wrapper_matched": True, "caveat": "within Qwen session; capped regimes are trace reintegrations"})
interactions = pd.DataFrame(interaction_rows)
for idx, row in interactions[interactions["ci95_low"].isna()].iterrows():
    model = row["model"]
    variants = ["C1", "C2", "C3", "C4"] if model == "llama" else ["Q1-W", "Q2", "Q3-Ref", "Q4-Ref"]
    subset = energy.query("model == @model and context == 4096 and regime == @row['regime'] and source_variant in @variants")
    pivot = subset.pivot(index="block", columns="source_variant", values=row["metric"])
    values = (pivot[variants[3]] - pivot[variants[2]]) - (pivot[variants[1]] - pivot[variants[0]])
    half = 2.7764451051977987 * float(values.std(ddof=1)) / math.sqrt(len(values))
    interactions.loc[idx, "ci95_low"] = float(values.mean() - half)
    interactions.loc[idx, "ci95_high"] = float(values.mean() + half)

def nondominated(energy_values, quality_values):
    n = len(energy_values)
    out = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i != j and energy_values[j] <= energy_values[i] and quality_values[j] >= quality_values[i] and (energy_values[j] < energy_values[i] or quality_values[j] > quality_values[i]):
                out[i] = False
                break
    return out

pareto_rows = []
for model, variants in [("llama", ["C0", "C1-W", "C2", "C3", "C4"]),
                        ("qwen", ["Q0", "Q1-W", "Q2", "Q3-Ref", "Q4-Ref"])]:
    point_energy, point_quality, point_memory = [], [], []
    for variant in variants:
        e = energy.query("model == @model and source_variant == @variant and context == 4096 and regime == 'raw'")
        q_variant = variant.replace("C3", "C3-Ref").replace("C4", "C4-Ref") if model == "llama" else variant
        q = hard.query("model == @model and source_variant == @q_variant")
        assert len(e) == 5 and len(q) == 60
        point_energy.append(e.total_energy_j.mean())
        point_quality.append(q.retrieval_correct.mean())
        point_memory.append(e.post_decode_cache_live_bytes.mean())
    membership = nondominated(np.array(point_energy), np.array(point_quality))
    frequencies = np.zeros(len(variants))
    rng = np.random.default_rng(BOOTSTRAP_SEED + (0 if model == "llama" else 1))
    energy_frames = [energy.query("model == @model and source_variant == @v and context == 4096 and regime == 'raw'") for v in variants]
    energy_groups = [frame.total_energy_j.to_numpy() for frame in energy_frames]
    energy_boot_keys = [
        (frame.source_session.iloc[0], "separate_q0_phase" if variant == "Q0" else "registered_session")
        for frame, variant in zip(energy_frames, variants)
    ]
    quality_groups = []
    for variant in variants:
        q_variant = variant.replace("C3", "C3-Ref").replace("C4", "C4-Ref") if model == "llama" else variant
        quality_groups.append(hard.query("model == @model and source_variant == @q_variant").set_index("scenario_id"))
    depths = sorted(hard.query("model == @model").depth.unique())
    for _ in range(BOOTSTRAP_REPS):
        idx_by_key = {key: rng.integers(0, 5, 5) for key in set(energy_boot_keys)}
        eboot = np.array([values[idx_by_key[key]].mean() for values, key in zip(energy_groups, energy_boot_keys)])
        scenario_ids = []
        for depth in depths:
            ids = sorted(hard.query("model == @model and depth == @depth").scenario_id.unique())
            scenario_ids.extend(rng.choice(ids, len(ids), replace=True))
        qboot = np.array([group.loc[scenario_ids].retrieval_correct.mean() for group in quality_groups])
        frequencies += nondominated(eboot, qboot)
    for i, variant in enumerate(variants):
        successes = int(round(point_quality[i] * 60))
        low, high = wilson(successes, 60)
        pareto_rows.append({"model": model, "quality_population": "hard60", "source_variant": variant,
            "normalized_role": ROLE[variant], "evidence_tier": "measured_system", "regime": "raw",
            "context": 4096, "energy_mean_j": point_energy[i],
            "energy_sd_j": float(energy_groups[i].std(ddof=1)), "persistent_kv_bytes_mean": point_memory[i],
            "retrieval_successes": successes, "retrieval_n": 60, "retrieval_estimate": point_quality[i],
            "retrieval_wilson_low": low, "retrieval_wilson_high": high,
            "point_frontier_member": bool(membership[i]), "bootstrap_repetitions": BOOTSTRAP_REPS,
            "bootstrap_frontier_membership_frequency": frequencies[i] / BOOTSTRAP_REPS,
            "frequency_interpretation": "resampling selection stability; not truth probability"})
empirical_pareto = pd.DataFrame(pareto_rows)

llama_stability = pd.read_csv(LLAMA_POWER / "ranking_stability.csv").assign(model="llama", sensitivity_type="trace_clipping_reintegration")
qwen_stability = pd.read_csv(QWEN / "analysis/ranking_stability.csv").assign(model="qwen", sensitivity_type="trace_clipping_reintegration")
power_stability = pd.concat([llama_stability, qwen_stability], ignore_index=True, sort=False)
llama_significance = pd.read_csv(LLAMA_POWER / "significance_stability.csv").assign(model="llama")
qwen_significance = pd.read_csv(QWEN / "analysis/significance_stability.csv").assign(model="qwen")
registered_significance = pd.concat([llama_significance, qwen_significance], ignore_index=True, sort=False)
llama_inference = pd.read_csv(LLAMA_POWER / "core_4096_tests_by_regime.csv").assign(model="llama")
qwen_inference = pd.read_csv(QWEN / "analysis/core_energy_tests_by_regime.csv").assign(model="qwen")
registered_inferential_tests = pd.concat([llama_inference, qwen_inference], ignore_index=True, sort=False)
for idx, effect in within_effects.iterrows():
    if effect["context"] != 4096 or not effect["design"].startswith("paired_within"):
        continue
    comparison_variant = {"C3": "C3-Ref", "C4": "C4-Ref"}.get(effect["comparison_variant"], effect["comparison_variant"])
    comparison = f"{effect['baseline_variant']}-{comparison_variant}"
    if effect["model"] == "llama":
        metric = effect["metric"]
        if metric in {"total_energy_j", "decode_energy_j"}:
            metric = metric.replace("_j", f"_{effect['regime']}_j")
        match = llama_inference.query("regime == @effect['regime'] and comparison == @comparison and metric == @metric")
    else:
        match = qwen_inference.query("regime == @effect['regime'] and comparison == @comparison and metric == @effect['metric']")
    if len(match) == 1:
        within_effects.loc[idx, "p_adjusted"] = float(match.iloc[0]["p_holm_20"])
        within_effects.loc[idx, "p_adjusted_source"] = "accepted registered within-session test"

quality_contrast_rows = []
rng_contrast = np.random.default_rng(BOOTSTRAP_SEED + 31)
model_effect_samples = {}
for model, baseline_variant, eviction_variant in [("llama", "C1-W", "C2"), ("qwen", "Q1-W", "Q2")]:
    model_hard = hard.query("model == @model")
    baseline_q = model_hard.query("source_variant == @baseline_variant").set_index("scenario_id")
    eviction_q = model_hard.query("source_variant == @eviction_variant").set_index("scenario_id")
    samples = []
    for _ in range(5000):
        ids = []
        for depth in sorted(model_hard.depth.unique()):
            pool = sorted(model_hard.query("depth == @depth").scenario_id.unique())
            ids.extend(rng_contrast.choice(pool, len(pool), replace=True))
        samples.append(float(eviction_q.loc[ids].retrieval_correct.mean() - baseline_q.loc[ids].retrieval_correct.mean()))
    model_effect_samples[model] = np.asarray(samples)
    estimate = float(eviction_q.retrieval_correct.mean() - baseline_q.retrieval_correct.mean())
    low, high = np.quantile(samples, [0.025, 0.975])
    quality_contrast_rows.append({"contrast": f"{model}_eviction_minus_awq", "model": model,
        "estimate": estimate, "bootstrap_ci95_low": low, "bootstrap_ci95_high": high,
        "bootstrap_repetitions": 5000, "paired_within_depth": True,
        "cross_model_pairing": False, "interpretation": "within-model hard60 retrieval-rate change"})
cross_samples = model_effect_samples["qwen"] - model_effect_samples["llama"]
cross_low, cross_high = np.quantile(cross_samples, [0.025, 0.975])
quality_contrast_rows.append({"contrast": "qwen_minus_llama_eviction_effect", "model": "cross_model_descriptive",
    "estimate": float((1/60-48/60) - (37/60-38/60)),
    "bootstrap_ci95_low": cross_low, "bootstrap_ci95_high": cross_high,
    "bootstrap_repetitions": 5000, "paired_within_depth": True,
    "cross_model_pairing": False,
    "interpretation": "independent model bootstraps; tokenizer-specific prompts; architecture alone not causal"})
cross_model_quality_contrasts = pd.DataFrame(quality_contrast_rows)
print("Empirical synthesis complete")


## Run 7 - Counterfactual eligibility and ideal layout assumptions

The ideal layout is a perfectly byte-packed persistent-KV lower envelope with
one-byte alignment and zero persistent metadata because dimensions and bit width
are treated as static kernel parameters. It is not a CUDA allocator model. Tensor
descriptors are reconstructed from runtime-asserted shape/dtype contracts and
persisted aggregate bytes; no live tensor dump was preserved.


In [ ]:
@dataclass(frozen=True)
class Geometry:
    model: str
    layers: int
    kv_heads: int
    head_dim: int

GEOMETRY = {
    "llama": Geometry("llama", 28, 8, 128),
    "qwen": Geometry("qwen", 36, 2, 128),
}

eligibility_rows = []
for model, prefix in [("llama", "C"), ("qwen", "Q")]:
    tq_labels = {2: ("C3-TQ2" if model == "llama" else "Q3-Ref-TQ2"),
                 3: ("C3-TQ3" if model == "llama" else "Q3-Ref-TQ3"),
                 4: ("C3" if model == "llama" else "Q3-Ref")}
    for bits in [2, 3, 4]:
        observed = True
        common = model == "qwen"
        hard_eligible = bits == 4
        eligibility_rows.append({"model": model, "source_variant": tq_labels[bits],
            "projected_variant": f"{model}_ideal_tq{bits}", "normalized_role": "awq_tq_ideal",
            "context": 4096, "average_budget": 1024, "nominal_tq_bits": bits, "has_eviction": False,
            "observed_numerical_path": observed, "observed_energy_path": observed,
            "observed_hard60_quality": hard_eligible, "observed_common15_quality": common,
            "storage_only_eligible": True,
            "quality_energy_pareto_eligible": hard_eligible or common,
            "quality_population": "hard60" if hard_eligible else ("qwen_common15" if common else "historical_easy_ineligible"),
            "baseline_variant": "C1" if model == "llama" else "Q1-W",
            "baseline_session": EXPECTED["llama_v5"]["session"] if model == "llama" else EXPECTED["qwen"]["session"],
            "eligibility_reason": "observed same-bit TQ-only numerical path" if (hard_eligible or common) else "Llama low-bit quality task is not hard60-comparable"})
        eligibility_rows.append({"model": model, "source_variant": ("C4" if model == "llama" else "Q4-Ref") if bits == 4 else f"unobserved_{prefix}4_TQ{bits}",
            "projected_variant": f"{model}_ideal_eviction_tq{bits}", "normalized_role": "awq_eviction_tq_ideal",
            "context": 4096, "average_budget": 1024, "nominal_tq_bits": bits, "has_eviction": True,
            "observed_numerical_path": bits == 4, "observed_energy_path": bits == 4,
            "observed_hard60_quality": bits == 4, "observed_common15_quality": False,
            "storage_only_eligible": True, "quality_energy_pareto_eligible": bits == 4,
            "quality_population": "hard60" if bits == 4 else "unknown",
            "baseline_variant": "C2" if model == "llama" else "Q2",
            "baseline_session": EXPECTED["llama_v5"]["session"] if model == "llama" else EXPECTED["qwen"]["session"],
            "eligibility_reason": "observed four-bit compound path" if bits == 4 else "unobserved low-bit compound; storage only"})
eligibility_rows.append({"model": "qwen", "source_variant": "Q3-Ref",
    "projected_variant": "qwen_ideal_tq4_common15", "normalized_role": "awq_tq_ideal",
    "context": 4096, "average_budget": 1024, "nominal_tq_bits": 4, "has_eviction": False,
    "observed_numerical_path": True, "observed_energy_path": True,
    "observed_hard60_quality": False, "observed_common15_quality": True,
    "storage_only_eligible": True, "quality_energy_pareto_eligible": True,
    "quality_population": "qwen_common15", "baseline_variant": "Q1-W",
    "baseline_session": EXPECTED["qwen"]["session"],
    "eligibility_reason": "observed four-bit TQ-only path restricted to the common 15 scenario IDs"})
eligibility = pd.DataFrame(eligibility_rows)
assert not eligibility.query("not observed_numerical_path and quality_energy_pareto_eligible").shape[0]

assumptions = {
    "schema_version": SCHEMA,
    "evidence_boundary": {
        "measured": "accepted FP16/AWQ/eviction/TQ-Ref systems rows and traces",
        "realized_reference": "numerically active uint8-indexed materialized-KV unfused TQ-Ref",
        "ideal": "analytical perfectly packed/fused persistent-KV reference envelope",
        "shape_provenance": "reconstructed_from_runtime_asserted_contract_and_persisted_aggregate_bytes",
    },
    "model_geometry": {name: vars(value) for name, value in GEOMETRY.items()},
    "storage_layout": {"batch": 1, "fp_bytes": FP_BYTES, "norm_bytes": NORM_BYTES,
        "residual_tokens": RESIDUAL_TOKENS, "alignment_bytes": ALIGNMENT_BYTES,
        "metadata_bytes_per_layer": METADATA_BYTES_PER_LAYER,
        "layout_claim": "perfect_byte_packed_reference_envelope_not_allocator_claim"},
    "phase_dataflow": {"prefill": "full prompt attention precedes eviction/quantized persistence",
        "decode": "logical compulsory reads/writes; ideal reconstruction remains in registers/shared memory",
        "hardware_dram_traffic_identified": False, "decode_tokens": DECODE_TOKENS},
    "energy_model": {"epsilon_mem_domain": None, "o_net_domain": None,
        "numerical_energy_projection_allowed": False,
        "equation": "E_ideal = E_baseline - epsilon_mem * Delta_D + O_net"},
    "quality_model": {"unchanged_quality_requires_strict_output_equivalence": True,
        "implementation_quality_change_outside_equivalence": "unidentified",
        "delta_impl_domain": "-wilson_low <= delta_impl <= 1-wilson_high so the full shifted quality interval remains in [0,1]"},
    "bootstrap": {"repetitions": BOOTSTRAP_REPS, "seed": BOOTSTRAP_SEED,
        "frequency_interpretation": "resampling selection stability only"},
    "excluded_claims": ["measured ideal kernel", "exact DRAM traffic", "total peak VRAM",
        "projected latency", "lower instantaneous power", "universal best configuration"],
}
print("Eligibility matrix and assumptions declared")


## Run 8 - Layerwise packed-memory ledger and unit gates

The FP formula must reproduce accepted retained-FP bytes. The same logical
formula at eight physical index bits must reproduce persisted TQ package bytes.
Only after those gates pass are nominal 2/3/4-bit packed footprints emitted.


In [ ]:
def build_budgets(layers, average_budget, window=32, beta=20):
    old_average = average_budget - window
    last = round(old_average / beta)
    first = 2 * old_average - last
    values = np.rint(np.linspace(first, last, layers)).astype(int) + window
    assert len(values) == layers and int(values.sum()) == layers * average_budget
    assert np.all(values[:-1] >= values[1:])
    return tuple(int(x) for x in values)

def retained_lengths(model, context, has_eviction, average_budget, phase):
    g = GEOMETRY[model]
    base = build_budgets(g.layers, average_budget) if has_eviction else (context,) * g.layers
    if phase == "decode":
        return tuple(x + DECODE_TOKENS for x in base)
    return base

def align_up(value, alignment):
    assert value >= 0 and alignment >= 1
    return ((value + alignment - 1) // alignment) * alignment

def packed_bytes(elements, bits):
    return (int(elements) * int(bits) + 7) // 8

def fp_kv_bytes(g, retained):
    return 2 * g.kv_heads * g.head_dim * FP_BYTES * sum(retained)

def tensor_ledger(model, projected_variant, context, average_budget, bits, has_eviction, phase):
    g = GEOMETRY[model]
    retained = retained_lengths(model, context, has_eviction, average_budget, phase)
    rows = []
    for layer, tokens in enumerate(retained):
        residual = min(RESIDUAL_TOKENS, tokens)
        old = tokens - residual
        for kv in ["K", "V"]:
            categories = [
                ("index", [1, g.kv_heads, old, g.head_dim], "packed_uint", old * g.kv_heads * g.head_dim, bits),
                ("norm", [1, g.kv_heads, old, 1], "float32", old * g.kv_heads, 8 * NORM_BYTES),
                ("residual", [1, g.kv_heads, residual, g.head_dim], "float16", residual * g.kv_heads * g.head_dim, 8 * FP_BYTES),
                ("metadata", [1], "static", 0, 8),
            ]
            for category, shape, dtype, elements, width in categories:
                raw = METADATA_BYTES_PER_LAYER if category == "metadata" and kv == "K" else packed_bytes(elements, width)
                rows.append({"model": model, "projected_variant": projected_variant,
                    "context": context, "phase": phase, "average_budget": average_budget,
                    "nominal_tq_bits": bits, "layer": layer, "kv_kind": kv,
                    "tensor_category": category, "shape_json": json.dumps(shape), "dtype": dtype,
                    "element_count": elements, "unaligned_bytes": raw,
                    "alignment_bytes": ALIGNMENT_BYTES, "aligned_bytes": align_up(raw, ALIGNMENT_BYTES),
                    "metadata_policy": "static_kernel_parameters_zero_persistent_bytes"})
    return pd.DataFrame(rows)

def logical_tq_bytes(g, retained, bits):
    total = 0
    for tokens in retained:
        residual = min(RESIDUAL_TOKENS, tokens)
        old = tokens - residual
        total += 2 * (packed_bytes(g.kv_heads * old * g.head_dim, bits)
                      + g.kv_heads * old * NORM_BYTES
                      + g.kv_heads * residual * g.head_dim * FP_BYTES)
    return total

calibration_rows = []
for model, frame, variants in [
    ("llama", energy.query("source_experiment == 'llama_v5' and regime == 'raw'"), ["C3", "C4"]),
    ("qwen", energy.query("source_experiment == 'qwen_full' and regime == 'raw'"), ["Q3-Ref", "Q4-Ref"]),
]:
    for variant in variants:
        source = frame.query("source_variant == @variant and context == 4096").iloc[0]
        evicted = "4" in variant
        for phase in ["prefill", "decode"]:
            retained = retained_lengths(model, 4096, evicted, 1024, phase)
            g = GEOMETRY[model]
            accepted_fp = int(source[f"post_{phase}_retained_fp16_bytes"])
            accepted_package = int(source[f"post_{phase}_tq_package_reported_bytes"])
            calculated_fp = fp_kv_bytes(g, retained)
            calculated_uint8 = logical_tq_bytes(g, retained, 8)
            ledger8 = tensor_ledger(model, f"{model}_uint8_calibration", 4096, 1024, 8, evicted, phase)
            category8 = ledger8.groupby("tensor_category")["aligned_bytes"].sum().to_dict()
            accepted_index = int(source[f"post_{phase}_tq_index_bytes"])
            accepted_norm = source.get(f"post_{phase}_tq_norm_bytes")
            norm_directly_persisted = model == "qwen" and accepted_norm is not None and not pd.isna(accepted_norm)
            accepted_norm = int(accepted_norm) if norm_directly_persisted else int(category8["norm"])
            accepted_logical_residual = accepted_package - accepted_index - accepted_norm
            accepted_materialized = int(source[f"post_{phase}_tq_materialized_bytes"])
            accepted_realized_residual = source.get(f"post_{phase}_tq_residual_bytes")
            accepted_realized_residual = (int(accepted_realized_residual)
                if accepted_realized_residual is not None and not pd.isna(accepted_realized_residual) else np.nan)
            assert calculated_fp == accepted_fp, (model, variant, phase, calculated_fp, accepted_fp)
            assert calculated_uint8 == accepted_package, (model, variant, phase, calculated_uint8, accepted_package)
            assert int(category8["index"]) == accepted_index, (model, variant, phase, "index", category8["index"], accepted_index)
            assert int(category8["norm"]) == accepted_norm, (model, variant, phase, "norm", category8["norm"], accepted_norm)
            assert int(category8["residual"]) == accepted_logical_residual, (model, variant, phase, "residual", category8["residual"], accepted_logical_residual)
            assert accepted_materialized == accepted_fp, (model, variant, phase, "materialized", accepted_materialized, accepted_fp)
            calibration_rows.append({"model": model, "variant": variant, "phase": phase,
                "accepted_retained_fp16_bytes": accepted_fp, "calculated_retained_fp16_bytes": calculated_fp,
                "accepted_uint8_package_bytes": accepted_package, "calculated_uint8_package_bytes": calculated_uint8,
                "accepted_index_bytes": accepted_index, "calculated_index_bytes_from_shape_dtype": int(category8["index"]),
                "accepted_norm_bytes": accepted_norm, "calculated_norm_bytes_from_shape_dtype": int(category8["norm"]),
                "accepted_logical_residual_bytes": accepted_logical_residual,
                "calculated_logical_residual_bytes_from_shape_dtype": int(category8["residual"]),
                "accepted_realized_residual_storage_bytes": accepted_realized_residual,
                "realized_residual_storage_covers_logical_payload": (accepted_realized_residual >= accepted_logical_residual) if not pd.isna(accepted_realized_residual) else np.nan,
                "accepted_materialized_bytes": accepted_materialized,
                "calculated_materialized_bytes": calculated_fp,
                "norm_category_directly_persisted": norm_directly_persisted,
                "shape_dtype_provenance": "runtime-asserted contract plus persisted category bytes; no live tensor dump",
                "passed": True})
hard_category_rows_validated = 0
for variant in ["C3-Ref", "C4-Ref"]:
    variant_rows = hard_quality.query("variant == @variant")
    state_summaries = {}
    for source in variant_rows.to_dict("records"):
        step_tokens = int(source["cache_min_after"] - source["cache_min_prefill"])
        retained = tuple(x + step_tokens for x in (build_budgets(GEOMETRY["llama"].layers, 1024) if variant == "C4-Ref" else (4096,) * GEOMETRY["llama"].layers))
        calculated_index = sum(2 * packed_bytes(GEOMETRY["llama"].kv_heads * max(0, tokens-RESIDUAL_TOKENS) * GEOMETRY["llama"].head_dim, 8) for tokens in retained)
        calculated_norm = sum(2 * GEOMETRY["llama"].kv_heads * max(0, tokens-RESIDUAL_TOKENS) * NORM_BYTES for tokens in retained)
        calculated_residual = sum(2 * GEOMETRY["llama"].kv_heads * min(tokens, RESIDUAL_TOKENS) * GEOMETRY["llama"].head_dim * FP_BYTES for tokens in retained)
        calculated_fp = fp_kv_bytes(GEOMETRY["llama"], retained)
        assert calculated_index == int(source["generation_final_tq_index_bytes"])
        assert calculated_norm == int(source["generation_final_tq_norm_bytes"])
        assert calculated_index + calculated_norm + calculated_residual == int(source["generation_final_tq_package_reported_bytes"])
        assert calculated_fp == int(source["generation_final_tq_materialized_bytes"])
        assert int(source["generation_final_tq_residual_bytes"]) >= calculated_residual
        hard_category_rows_validated += 1
        state = (step_tokens, calculated_index, calculated_norm, calculated_residual, calculated_fp,
                 int(source["generation_final_tq_residual_bytes"]))
        state_summaries[state] = state_summaries.get(state, 0) + 1
    for state, scenario_count in sorted(state_summaries.items()):
        step_tokens, calculated_index, calculated_norm, calculated_residual, calculated_fp, realized_residual = state
        calibration_rows.append({"model": "llama", "variant": variant,
            "phase": f"hard_generation_final_plus_{step_tokens}", "scenario_rows_validated": scenario_count,
            "accepted_retained_fp16_bytes": calculated_fp, "calculated_retained_fp16_bytes": calculated_fp,
            "accepted_uint8_package_bytes": calculated_index + calculated_norm + calculated_residual,
            "calculated_uint8_package_bytes": calculated_index + calculated_norm + calculated_residual,
            "accepted_index_bytes": calculated_index, "calculated_index_bytes_from_shape_dtype": calculated_index,
            "accepted_norm_bytes": calculated_norm, "calculated_norm_bytes_from_shape_dtype": calculated_norm,
            "accepted_logical_residual_bytes": calculated_residual,
            "calculated_logical_residual_bytes_from_shape_dtype": calculated_residual,
            "accepted_realized_residual_storage_bytes": realized_residual,
            "realized_residual_storage_covers_logical_payload": True,
            "accepted_materialized_bytes": calculated_fp, "calculated_materialized_bytes": calculated_fp,
            "norm_category_directly_persisted": True,
            "shape_dtype_provenance": "all accepted hard-run category rows reconciled to runtime-asserted shape/dtype contract; no live tensor dump",
            "passed": True})
assert hard_category_rows_validated == 120
storage_calibration = pd.DataFrame(calibration_rows)

ledger_frames, footprint_rows = [], []
for row in eligibility.to_dict("records"):
    for phase in ["prefill", "decode"]:
        ledger = tensor_ledger(row["model"], row["projected_variant"], row["context"],
            row["average_budget"], row["nominal_tq_bits"], row["has_eviction"], phase)
        ledger_frames.append(ledger)
        totals = ledger.groupby("tensor_category")["aligned_bytes"].sum().to_dict()
        retained = retained_lengths(row["model"], row["context"], row["has_eviction"], row["average_budget"], phase)
        fp_bytes = fp_kv_bytes(GEOMETRY[row["model"]], retained)
        ideal = int(ledger["aligned_bytes"].sum())
        index_savings = fp_bytes - totals.get("index", 0) - totals.get("residual", 0)
        overhead = totals.get("norm", 0) + totals.get("metadata", 0)
        footprint_rows.append({**{key: row[key] for key in ["model", "projected_variant", "context", "average_budget", "nominal_tq_bits", "has_eviction", "storage_only_eligible", "quality_energy_pareto_eligible"]},
            "phase": phase, "retained_tokens_sum": sum(retained),
            "quantized_tokens_sum": sum(max(0, x-RESIDUAL_TOKENS) for x in retained),
            "residual_tokens_sum": sum(min(x, RESIDUAL_TOKENS) for x in retained),
            "fp_logical_bytes": fp_bytes, "ideal_index_bytes": totals.get("index", 0),
            "ideal_norm_bytes": totals.get("norm", 0), "ideal_residual_bytes": totals.get("residual", 0),
            "ideal_metadata_bytes": totals.get("metadata", 0), "ideal_logical_bytes": ideal,
            "ideal_aligned_bytes": ideal, "alignment_bytes": ALIGNMENT_BYTES,
            "decimal_mb": ideal / 1e6, "mib": ideal / 2**20,
            "fp_bytes_per_retained_token_layer": 2 * GEOMETRY[row["model"]].kv_heads * GEOMETRY[row["model"]].head_dim * FP_BYTES,
            "ideal_bytes_per_retained_token_layer": ideal / sum(retained),
            "geometry_normalized_ideal_to_fp_ratio": ideal / fp_bytes,
            "gross_index_savings_bytes": index_savings, "extra_overhead_bytes": overhead,
            "persistent_memory_dominance_proved": ideal < fp_bytes,
            "proof_condition": "packed index savings exceed norms, residual representation difference, alignment, and metadata"})
tensor_accounting = pd.concat(ledger_frames, ignore_index=True)
footprints = pd.DataFrame(footprint_rows)

for (model, evicted, phase), group in footprints.groupby(["model", "has_eviction", "phase"]):
    values = group.groupby("nominal_tq_bits").ideal_aligned_bytes.first()
    assert values[2] <= values[3] <= values[4]
for model in GEOMETRY:
    low = footprints.query("model == @model and has_eviction and nominal_tq_bits == 4 and phase == 'prefill'").ideal_aligned_bytes.iloc[0]
    high_budget = logical_tq_bytes(GEOMETRY[model], build_budgets(GEOMETRY[model].layers, 1536), 4)
    assert low <= high_budget
assert footprints["persistent_memory_dominance_proved"].all()
print("FP, uint8 payload, monotonicity, and memory-dominance gates passed")


## Run 9 - Measured, realized-reference, and ideal storage

Realized-reference live storage includes materialized K/V. Ideal storage removes
that materialization only in the explicit counterfactual tier. Persistent KV is
not equated with total peak VRAM.


In [ ]:
storage_rows = []
for eligible in eligibility.query("observed_numerical_path").to_dict("records"):
    source_variant = eligible["source_variant"]
    el_model = eligible["model"]
    el_baseline_variant = eligible["baseline_variant"]
    el_projected_variant = eligible["projected_variant"]
    source = energy.query("model == @el_model and source_variant == @source_variant and context == 4096 and regime == 'raw'")
    if source.empty:
        continue
    baseline = energy.query("model == @el_model and source_variant == @el_baseline_variant and context == 4096 and regime == 'raw'")
    assert not baseline.empty
    for phase in ["prefill", "decode"]:
        fp = footprints.query("projected_variant == @el_projected_variant and phase == @phase").iloc[0]
        storage_rows.append({"model": eligible["model"], "source_variant": source_variant,
            "projected_variant": eligible["projected_variant"], "context": 4096, "phase": phase,
            "nominal_tq_bits": eligible["nominal_tq_bits"], "baseline_variant": eligible["baseline_variant"],
            "evidence_tier": "measured_realized_and_ideal_side_by_side",
            "measured_non_tq_live_bytes": baseline[f"post_{phase}_cache_live_bytes"].mean(),
            "measured_non_tq_retained_fp_bytes": baseline[f"post_{phase}_retained_fp16_bytes"].mean(),
            "realized_ref_live_bytes": source[f"post_{phase}_cache_live_bytes"].mean(),
            "realized_ref_materialized_bytes": source[f"post_{phase}_tq_materialized_bytes"].mean(),
            "realized_ref_uint8_index_bytes": source[f"post_{phase}_tq_index_bytes"].mean(),
            "realized_ref_logical_payload_bytes": source[f"post_{phase}_tq_package_reported_bytes"].mean(),
            "ideal_logical_bytes": fp["ideal_logical_bytes"], "ideal_aligned_bytes": fp["ideal_aligned_bytes"],
            "realized_minus_ideal_bytes": source[f"post_{phase}_cache_live_bytes"].mean() - fp["ideal_aligned_bytes"],
            "ideal_vs_fp_ratio": fp["ideal_aligned_bytes"] / fp["fp_logical_bytes"],
            "fp_bytes_per_retained_token_layer": fp["fp_logical_bytes"] / fp["retained_tokens_sum"],
            "realized_live_bytes_per_retained_token_layer": source[f"post_{phase}_cache_live_bytes"].mean() / fp["retained_tokens_sum"],
            "ideal_bytes_per_retained_token_layer": fp["ideal_aligned_bytes"] / fp["retained_tokens_sum"],
            "geometry_normalized_ideal_to_fp_ratio": fp["ideal_aligned_bytes"] / fp["fp_logical_bytes"],
            "decimal_mb": fp["ideal_aligned_bytes"] / 1e6, "mib": fp["ideal_aligned_bytes"] / 2**20,
            "total_peak_vram_modeled": False,
            "shape_provenance": "reconstructed_from_runtime_asserted_contract_and_persisted_aggregate_bytes"})
storage_comparison = pd.DataFrame(storage_rows)
assert not storage_comparison["total_peak_vram_modeled"].any()
print("Storage tiers separated")


## Run 10 - Prefill and fixed-decode logical byte demand

Prefill excludes full-prompt attention reads because all paths attend before
eviction. Decode counts growing K/V reads, packed indices, norms, FP residuals,
new-token writes, and residual-to-packed transitions. These are exact counts
under the declared algorithmic dataflow, not measured DRAM traffic.


In [ ]:
def ideal_compulsory_write_bytes(g, retained, bits):
    return logical_tq_bytes(g, retained, bits)

def ideal_compulsory_read_bytes(g, retained, bits):
    return logical_tq_bytes(g, retained, bits)

def phase_demand(eligible):
    model, bits = eligible["model"], int(eligible["nominal_tq_bits"])
    g = GEOMETRY[model]
    el_projected_variant = eligible["projected_variant"]
    prefill = retained_lengths(model, eligible["context"], eligible["has_eviction"], eligible["average_budget"], "prefill")
    full_prompt_attention_read = fp_kv_bytes(g, (eligible["context"],) * g.layers)
    ideal_prefill_compulsory = ideal_compulsory_write_bytes(g, prefill, bits)
    ideal_prefill_zero_traffic = ideal_prefill_compulsory
    ideal_prefill_non_traffic_overhead = max(0, fp_kv_bytes(g, prefill) - ideal_prefill_zero_traffic)
    ideal_prefill_retained_read = ideal_compulsory_read_bytes(g, prefill, bits)
    ideal_prefill_compulsory_read_bytes = full_prompt_attention_read + ideal_prefill_retained_read
    prefill_delta = full_prompt_attention_read - (ideal_prefill_zero_traffic + ideal_prefill_retained_read)

    decode = retained_lengths(model, eligible["context"], eligible["has_eviction"], eligible["average_budget"], "decode")
    ideal_decode_read = ideal_compulsory_read_bytes(g, decode, bits)
    ideal_decode_compulsory = ideal_compulsory_write_bytes(g, [1] * g.layers, bits) + ideal_decode_read
    ideal_decode_zero_traffic = ideal_decode_compulsory
    ideal_decode_non_traffic_overhead = max(0, fp_kv_bytes(g, decode) + fp_kv_bytes(g, [1] * g.layers) - ideal_decode_zero_traffic)
    ideal_decode_fp_baseline_read = fp_kv_bytes(g, decode)
    decode_delta = ideal_decode_fp_baseline_read - ideal_decode_read

    total_ideal_compulsory = ideal_prefill_compulsory + ideal_decode_compulsory
    total_fp_read_write = full_prompt_attention_read + fp_kv_bytes(g, prefill) + ideal_decode_fp_baseline_read + fp_kv_bytes(g, [1] * g.layers)
    total_delta = total_fp_read_write - total_ideal_compulsory

    common = {"model": model, "projected_variant": eligible["projected_variant"],
        "baseline_variant": eligible["baseline_variant"], "source_session": eligible["baseline_session"],
        "context": eligible["context"], "average_budget": eligible["average_budget"],
        "nominal_tq_bits": bits, "has_eviction": eligible["has_eviction"],
        "ideal_tq_packing_assumed": True, "ideal_quant_kernel_fusion_assumed": True,
        "modeled_compulsory_algorithmic_byte_demand_only": True, "hardware_dram_traffic_modeled": False,
        "prefill_compulsory_model": "full_prompt_attention_read_plus_ideal_compressed_write_and_retained_compressed_read"}

    return [
        {**common, "phase": "prefill",
         "fp16_baseline_bytes_for_row": full_prompt_attention_read + fp_kv_bytes(g, prefill),
         "ideal_compulsory_read_bytes": ideal_prefill_compulsory_read_bytes,
         "ideal_compulsory_write_bytes": ideal_prefill_compulsory,
         "ideal_compulsory_total_bytes": ideal_prefill_compulsory_read_bytes + ideal_prefill_compulsory,
         "ideal_prefill_index_write_bytes": int(footprints.query("projected_variant == @el_projected_variant and phase == 'prefill'").ideal_index_bytes.iloc[0]),
         "ideal_prefill_norm_write_bytes": int(footprints.query("projected_variant == @el_projected_variant and phase == 'prefill'").ideal_norm_bytes.iloc[0]),
         "ideal_prefill_residual_write_bytes": int(footprints.query("projected_variant == @el_projected_variant and phase == 'prefill'").ideal_residual_bytes.iloc[0]),
         "ideal_non_traffic_overhead_bytes": ideal_prefill_non_traffic_overhead,
         "delta_d_bytes_for_row": prefill_delta, "delta_positive": prefill_delta > 0},
        {**common, "phase": "decode",
         "fp16_baseline_bytes_for_row": ideal_decode_fp_baseline_read,
         "ideal_compulsory_read_bytes": ideal_decode_read,
         "ideal_compulsory_write_bytes": ideal_compulsory_write_bytes(g, [1] * g.layers, bits),
         "ideal_compulsory_total_bytes": ideal_decode_compulsory,
         "ideal_prefill_index_write_bytes": np.nan, "ideal_prefill_norm_write_bytes": np.nan, "ideal_prefill_residual_write_bytes": np.nan,
         "ideal_non_traffic_overhead_bytes": ideal_decode_non_traffic_overhead,
         "delta_d_bytes_for_row": decode_delta, "delta_positive": decode_delta > 0},
        {**common, "phase": "total",
         "fp16_baseline_bytes_for_row": total_fp_read_write,
         "ideal_compulsory_read_bytes": ideal_prefill_compulsory_read_bytes + ideal_decode_read,
         "ideal_compulsory_write_bytes": ideal_prefill_compulsory + ideal_compulsory_write_bytes(g, [1] * g.layers, bits),
         "ideal_compulsory_total_bytes": total_ideal_compulsory,
         "ideal_prefill_index_write_bytes": np.nan, "ideal_prefill_norm_write_bytes": np.nan, "ideal_prefill_residual_write_bytes": np.nan,
         "ideal_non_traffic_overhead_bytes": ideal_prefill_non_traffic_overhead + ideal_decode_non_traffic_overhead,
         "delta_d_bytes_for_row": total_delta, "delta_positive": total_delta > 0},
    ]

demand_rows = []
for eligible in eligibility.query("observed_numerical_path").to_dict("records"):
    demand_rows.extend(phase_demand(eligible))
demand = pd.DataFrame(demand_rows)

assert demand.query("phase in ['decode', 'total']")["delta_positive"].all()
demand["phase_model_passed"] = True
print("Compulsory byte-demand model constructed")


## Run 11 - Symbolic energy break-even surfaces

For each observed same-bit path, the break-even condition is
`O_net < epsilon_mem * Delta_D`. `epsilon_mem >= 0`, `Delta_D > 0`, and
`E_ideal >= 0` are mandatory. No numerical ideal joules or numerical Pareto
coordinate is emitted.


In [ ]:
break_rows = []
for eligible in eligibility.query("observed_energy_path").to_dict("records"):
    el_model = eligible["model"]
    el_source_variant = eligible["source_variant"]
    el_baseline_variant = eligible["baseline_variant"]
    el_projected_variant = eligible["projected_variant"]
    source = energy.query("model == @el_model and source_variant == @el_source_variant and context == 4096")
    baseline = energy.query("model == @el_model and source_variant == @el_baseline_variant and context == 4096")
    for phase in ["prefill", "decode", "total"]:
        for regime in ["raw", "derived_c1_matched"]:
            b = baseline.query("regime == @regime")
            if b.empty:
                continue
            delta = int(demand.query("projected_variant == @el_projected_variant and phase == @phase").delta_d_bytes_for_row.iloc[0])
            break_rows.append({"model": eligible["model"], "projected_variant": eligible["projected_variant"],
                "source_variant": eligible["source_variant"], "baseline_variant": eligible["baseline_variant"],
                "source_session": eligible["baseline_session"], "context": 4096,
                "average_budget": eligible["average_budget"], "nominal_tq_bits": eligible["nominal_tq_bits"],
                "phase": phase, "regime": regime, "delta_d_bytes": delta, "break_even_available": delta > 0,
                "break_even_epsilon_expression": f"O_net / {delta}" if delta > 0 else np.nan,
                "break_even_o_net_expression": f"epsilon_mem * {delta}" if delta > 0 else np.nan,
                "ideal_tq_packing_assumed": True, "ideal_quant_fusion_assumed": True,
                "numerical_parameter_domain_declared": False, "numerical_ideal_energy_j": np.nan,
                "numerical_break_even_epsilon_j_per_byte": np.nan, "numerical_break_even_o_net_bytes": np.nan,
                "numerical_pareto_eligible": False,
                "cross_session_caveat": "Llama hard quality and historical energy are separate sessions" if eligible["model"] == "llama" else "none"})
break_even = pd.DataFrame(break_rows)
assert break_even.query("break_even_available")["delta_d_bytes"].gt(0).all()
assert break_even.query("not break_even_available")["break_even_epsilon_expression"].isna().all()
print("Break-even expressions computed")


## Run 12 - Full quality-threshold axis and conditional optimality

The threshold axis is partitioned exactly at every estimate and Wilson bound.
Point, conservative, and possible feasibility remain separate. Outside strict
output equivalence, implementation-induced quality change is an unidentified
structural parameter.


In [ ]:
def mean_std_wilson(series):
    series = pd.Series(series).dropna()
    n = len(series)
    if n == 0:
        return np.nan, np.nan, np.nan, np.nan
    m = float(series.mean())
    s = float(series.std(ddof=1)) if n > 1 else 0.0
    succ = int(round(series.sum()))
    w_low, w_high = wilson(succ, n)
    return m, s, w_low, w_high

projection_quality = []
for eligible in eligibility.query("quality_energy_pareto_eligible").to_dict("records"):
    el_model = eligible["model"]
    if eligible["quality_population"] == "hard60":
        quality_variant = eligible["source_variant"].replace("C3", "C3-Ref").replace("C4", "C4-Ref") if eligible["model"] == "llama" else eligible["source_variant"]
        row = quality_intervals.query("model == @el_model and quality_population == 'hard60' and source_variant == @quality_variant and depth == 'all'")
    elif eligible["quality_population"] == "qwen_common15":
        quality_variant = eligible["source_variant"]
        row = quality_intervals.query("model == 'qwen' and quality_population == 'qwen_common15' and source_variant == @quality_variant and depth == 'all'")
    else:
        raise ValueError(eligible)
    assert len(row) == 1, (eligible, row)
    row = row.iloc[0]
    q_est = float(row["estimate"])
    q_n = float(row["n"])
    q_std = float(np.sqrt(q_est * (1.0 - q_est) / q_n)) if q_n > 0 else 0.0
    projection_quality.append({**eligible, "quality_evidence_variant": quality_variant,
        "quality_mean": q_est, "quality_std": q_std, "wilson_low": float(row["wilson95_low"]), "wilson_high": float(row["wilson95_high"])})
projection_quality = pd.DataFrame(projection_quality)

common_ids = sorted(set.intersection(*(set(hard.query("model == 'qwen' and source_variant == @v")["scenario_id"])
    for v in ["C0", "C1", "C2", "Q3-Ref", "Q4-Ref", "Qwen2.5-3B-Instruct", "Qwen2.5-3B-Instruct-AWQ", "Qwen2.5-3B-Instruct-AWQ-PyramidKV-1024", "Qwen2.5-3B-Instruct-AWQ-TurboQuant-Ref-3bit"] if v in hard.query("model == 'qwen'")["source_variant"].unique())))
qwen_common15_rows = []
for variant in ["C0", "C1", "C2", "Q3-Ref", "Q4-Ref"]:
    qgroup = hard.query("model == 'qwen' and source_variant == @variant and scenario_id in @common_ids")
    col = "retrieval_correct" if "retrieval_correct" in qgroup else "score"
    qmean, qstd, qlow, qhigh = mean_std_wilson(qgroup[col])
    qbits = 4 if variant in ["C0", "C1", "C2"] else (3 if variant == "Q3-Ref" else 4)
    egroup = energy.query("model == 'qwen' and source_variant == @variant and context == 4096 and regime == 'raw'")
    emean, estd = egroup["total_energy_j"].mean(), egroup["total_energy_j"].std()
    qwen_common15_rows.append({"model": "qwen", "quality_population": "qwen_common15", "source_variant": variant,
        "quality_evidence_variant": variant, "nominal_tq_bits": qbits, "has_eviction": "Pyramid" in variant or variant in ["C2", "Q4-Ref"],
        "observed_numerical_path": True, "observed_energy_path": True, "quality_energy_pareto_eligible": True,
        "quality_mean": qmean, "quality_std": qstd, "wilson_low": qlow, "wilson_high": qhigh,
        "baseline_energy_mean_j": emean, "baseline_energy_std_j": estd})

quality_population_points = []
for row in projection_quality.to_dict("records"):
    row_proj = row["projected_variant"]
    row_model = row["model"]
    row_source = row["source_variant"]
    quality_population_points.append({"point_id": f"measured_{row['model']}_{row['source_variant']}",
        "model": row["model"], "quality_population": row["quality_population"], "source_variant": row["source_variant"],
        "projected_variant": row["projected_variant"], "evidence_tier": "measured_system",
        "wilson_low": row["wilson_low"], "wilson_high": row["wilson_high"], "quality_mean": row["quality_mean"],
        "persistent_kv_bytes": storage_comparison.query("projected_variant == @row_proj and phase == 'decode'").iloc[0]["realized_ref_live_bytes"],
        "baseline_energy_mean_j": break_even.query("projected_variant == @row_proj and phase == 'total' and regime == 'raw'").iloc[0]["delta_d_bytes"],
        "delta_d_bytes": break_even.query("projected_variant == @row_proj and phase == 'total' and regime == 'raw'").iloc[0]["delta_d_bytes"],
        "measured_energy_j": energy.query("model == @row_model and source_variant == @row_source and context == 4096 and regime == 'raw'")["total_energy_j"].mean() if row["model"] == "qwen" else energy.query("model == 'llama' and source_variant == @row_source and context == 4096 and regime == 'raw'")["total_energy_j"].mean()})

atomic_endpoints = sorted(set([0.0, 1.0] + list(projection_quality["wilson_low"]) + list(projection_quality["wilson_high"])))
thresholds = []
for i in range(len(atomic_endpoints) - 1):
    low, high = atomic_endpoints[i], atomic_endpoints[i+1]
    if low < high:
        thresholds.append((f"point-{i:02d}", low, high))

quality_regions = pd.DataFrame([{"a_min_region_id": rid, "a_min_low": low, "a_min_high": high} for rid, low, high in thresholds])

feasibility_rules = [("strict_upper_bound", "a_min_high"), ("conservative_lower_bound", "a_min_low")]
quality_scenarios = [("strict_output_equivalence", "measured_only"), ("projected_wilson_interval", "wilson_interval")]
energy_scenarios = [("measured_only_symbolic", "ideal_energy_undefined"), ("zero_net_overhead_symbolic", "ideal_energy_equals_baseline_minus_epsilon_delta")]

conditional_rows = []
robust_rows = []

for model in ["llama", "qwen"]:
    pop_sub = projection_quality.query("model == @model")
    for population in pop_sub["quality_population"].unique():
        pop_rows = projection_quality.query("model == @model and quality_population == @population")
        for region_id, low, high in thresholds:
            for rule_name, rule_bound in feasibility_rules:
                a_min = high if rule_bound == "a_min_high" else low
                for quality_scenario, quality_mode in quality_scenarios:
                    for scenario, energy_mode in energy_scenarios:
                        expression_key = "conditional_energy_expression"
                        c1_sub = pop_rows.query("source_variant in ['C1', 'C1-W', 'Q1', 'Q1-W', 'Qwen2.5-3B-Instruct-AWQ']")
                        c1 = c1_sub.iloc[0] if not c1_sub.empty else pop_rows.iloc[0]
                        c0_sub = pop_rows.query("source_variant in ['C0', 'Q0', 'Qwen2.5-3B-Instruct']")
                        c0 = c0_sub.iloc[0] if not c0_sub.empty else pop_rows.iloc[0]
                        c1_q_cond = "TRUE" if c1["wilson_low"] >= a_min else ("FALSE" if c1["wilson_high"] < a_min else f"({c1['wilson_low']} <= a_min <= {c1['wilson_high']})")
                        
                        candidates = []
                        for row in pop_rows.to_dict("records"):
                            row_proj = row["projected_variant"]
                            row_model = row["model"]
                            row_source = row["source_variant"]
                            footprint_storage = storage_comparison.query("projected_variant == @row_proj and phase == 'decode'").iloc[0]
                            footprint_ideal = footprints.query("projected_variant == @row_proj and phase == 'decode'").iloc[0]
                            be = break_even.query("projected_variant == @row_proj and phase == 'total' and regime == 'raw'").iloc[0]
                            e_meas = energy.query("model == @row_model and source_variant == @row_source and context == 4096 and regime == 'raw'")["total_energy_j"].mean()
                            
                            candidates.append({
                                "point_id": f"measured_{model}_{row['source_variant']}", "source_variant": row["source_variant"], "projected_variant": row["projected_variant"],
                                "evidence_tier": "measured_system", "wilson_low": row["wilson_low"], "wilson_high": row["wilson_high"],
                                "persistent_kv_bytes": footprint_storage["realized_ref_live_bytes"], "baseline_energy_mean_j": e_meas,
                                "delta_d_bytes": be["delta_d_bytes"], "conditional_energy_expression": f"{e_meas:.6f}"
                            })
                            if row["nominal_tq_bits"] in [2, 3, 4]:
                                candidates.append({
                                    "point_id": f"realized_ref_{model}_{row['source_variant']}", "source_variant": row["source_variant"], "projected_variant": row["projected_variant"],
                                    "evidence_tier": "realized_reference", "wilson_low": row["wilson_low"], "wilson_high": row["wilson_high"],
                                    "persistent_kv_bytes": footprint_storage["realized_ref_live_bytes"], "baseline_energy_mean_j": e_meas,
                                    "delta_d_bytes": be["delta_d_bytes"], "conditional_energy_expression": f"{e_meas:.6f} + O_net * e_joule_per_byte"
                                })
                                candidates.append({
                                    "point_id": f"ideal_{model}_{row['source_variant']}", "source_variant": row["source_variant"], "projected_variant": row["projected_variant"],
                                    "evidence_tier": "ideal_counterfactual", "wilson_low": row["wilson_low"], "wilson_high": row["wilson_high"],
                                    "persistent_kv_bytes": footprint_ideal["ideal_aligned_bytes"], "baseline_energy_mean_j": e_meas,
                                    "delta_d_bytes": be["delta_d_bytes"], "conditional_energy_expression": f"{e_meas:.6f} - ({be['delta_d_bytes']} * epsilon_mem)"
                                })

                        candidate_quality = {}
                        for point in candidates:
                            if quality_mode == "measured_only":
                                admitted = point["wilson_low"] >= a_min if point["evidence_tier"] != "ideal_counterfactual" else None
                                q_cond = "TRUE" if point["wilson_low"] >= a_min else "FALSE"
                                state = "admitted" if admitted else ("excluded" if admitted is False else "conditional")
                            else:
                                if point["evidence_tier"] == "ideal_counterfactual":
                                    q_cond = f"(delta_impl >= {a_min - point['wilson_high']:.6f})"
                                    admitted = None
                                    state = "conditional"
                                else:
                                    admitted = point["wilson_low"] >= a_min
                                    q_cond = "TRUE" if admitted else "FALSE"
                                    state = "admitted" if admitted else "excluded"
                            candidate_quality[point["point_id"]] = (state, q_cond, admitted)

                        potentially_admitted = [p for p in candidates if candidate_quality[p["point_id"]][2] is not False]
                        for point in candidates:
                            point_state, point_quality_condition, point_admitted = candidate_quality[point["point_id"]]
                            conditions = []
                            universal_dominator = False
                            for competitor in potentially_admitted:
                                if competitor["point_id"] == point["point_id"] or competitor["persistent_kv_bytes"] > point["persistent_kv_bytes"]:
                                    continue
                                operator = "<" if competitor["persistent_kv_bytes"] == point["persistent_kv_bytes"] else "<="
                                energy_condition = f"({competitor[expression_key]}) {operator} ({point[expression_key]})"
                                fixed_energy = competitor["evidence_tier"] != "ideal_counterfactual" and point["evidence_tier"] != "ideal_counterfactual"
                                if fixed_energy:
                                    condition_true = (competitor["baseline_energy_mean_j"] < point["baseline_energy_mean_j"] if operator == "<" else competitor["baseline_energy_mean_j"] <= point["baseline_energy_mean_j"])
                                    energy_condition = "always" if condition_true else "never"
                                    universal_dominator = universal_dominator or (condition_true and candidate_quality[competitor["point_id"]][1] == "TRUE")
                                zero_ideal_dominates_measured = (scenario == "zero_net_overhead_symbolic"
                                    and competitor["evidence_tier"] == "ideal_counterfactual"
                                    and point["evidence_tier"] != "ideal_counterfactual"
                                    and competitor["baseline_energy_mean_j"] <= point["baseline_energy_mean_j"]
                                    and competitor["delta_d_bytes"] > 0
                                    and candidate_quality[competitor["point_id"]][1] == "TRUE")
                                universal_dominator = universal_dominator or zero_ideal_dominates_measured
                                competitor_quality_condition = candidate_quality[competitor["point_id"]][1]
                                combined = energy_condition if competitor_quality_condition == "TRUE" else (
                                    "never" if competitor_quality_condition == "FALSE" or energy_condition == "never" else
                                    f"({competitor_quality_condition}) AND ({energy_condition})")
                                conditions.append({"competitor": competitor["point_id"], "memory_relation": operator,
                                    "quality_admission_condition": competitor_quality_condition,
                                    "energy_dominance_condition": energy_condition,
                                    "combined_dominance_condition": combined,
                                    "universal_under_declared_constraints": zero_ideal_dominates_measured or (fixed_energy and energy_condition == "always" and competitor_quality_condition == "TRUE")})
                            active = [c["combined_dominance_condition"] for c in conditions if c["combined_dominance_condition"] != "never"]
                            if point_admitted is False:
                                membership = "FALSE (quality excluded)"
                            elif universal_dominator:
                                membership = "FALSE (universal dominator under declared constraints)"
                            else:
                                nondominance = "TRUE" if not active else "NOT(" + " OR ".join(active) + ")"
                                membership = nondominance if point_quality_condition == "TRUE" else f"({point_quality_condition}) AND ({nondominance})"
                            assert "NOT()" not in membership
                            strict_minimum = point_admitted is True and all(point["persistent_kv_bytes"] < other["persistent_kv_bytes"]
                                for other in potentially_admitted if other["point_id"] != point["point_id"])
                            status = "excluded" if point_admitted is False else ("dominated_all_parameters" if universal_dominator else ("frontier_all_parameters" if strict_minimum else "frontier_conditional_region"))
                            conditional_rows.append({"model": model, "quality_population": population,
                                "a_min_region_id": region_id, "a_min_low": low, "a_min_high": high,
                                "feasibility_rule": rule_name, "quality_realization_scenario": quality_scenario,
                                "realization_scenario": scenario, "point_id": point["point_id"],
                                "source_variant": point["source_variant"], "projected_variant": point.get("projected_variant", ""),
                                "evidence_tier": point["evidence_tier"], "quality_admission_state": point_state,
                                "quality_admitted": point_admitted, "quality_admission_condition": point_quality_condition,
                                "quality_delta_lower_bound": -point["wilson_low"] if point["evidence_tier"] == "ideal_counterfactual" else np.nan,
                                "quality_delta_upper_bound": 1-point["wilson_high"] if point["evidence_tier"] == "ideal_counterfactual" else np.nan,
                                "persistent_kv_bytes": point["persistent_kv_bytes"],
                                "baseline_energy_mean_j": point["baseline_energy_mean_j"], "delta_d_bytes": point["delta_d_bytes"],
                                "conditional_energy_expression": point[expression_key],
                                "pairwise_dominance_conditions_json": json.dumps(conditions, sort_keys=True),
                                "pareto_membership_condition": membership, "symbolic_frontier_status": status,
                                "numerical_frontier_computable": False,
                                "bootstrap_frontier_membership_frequency": np.nan,
                                "status_reason": "exact symbolic quality and energy region; no arbitrary parameter grid"})
                            robust_class = "none_quality_excluded" if point_admitted is False else ("none_universally_dominated" if universal_dominator else ("all_by_strict_memory_minimum" if strict_minimum else "some_conditional_region"))
                            robust_rows.append({"model": model, "quality_population": population,
                                "a_min_region_id": region_id, "feasibility_rule": rule_name,
                                "quality_realization_scenario": quality_scenario, "realization_scenario": scenario,
                                "point_id": point["point_id"], "projected_variant": point.get("projected_variant", ""),
                                "evidence_tier": point["evidence_tier"],
                                "parameter_region_id": "epsilon_nonnegative_energy_nonnegative_quality_delta_symbolic",
                                "parameter_region_bounded": False, "strict_memory_minimum": strict_minimum,
                                "structurally_nondominated_for_all_energy_values": strict_minimum,
                                "robust_membership_class": robust_class, "membership_condition": membership,
                                "bootstrap_frequency_min": np.nan, "bootstrap_frequency_max": np.nan,
                                "frequency_bounds_available": False,
                                "reason": "all/some/none simplified under epsilon>=0, E>=0, O_net scenario, and quality admission"})
conditional_pareto = pd.DataFrame(conditional_rows)
robust_regions = pd.DataFrame(robust_rows)

assert conditional_pareto["pairwise_dominance_conditions_json"].map(json.loads).map(lambda x: isinstance(x, list)).all()
assert {"measured_system", "realized_reference", "ideal_counterfactual"}.issubset(set(conditional_pareto["evidence_tier"]))
assert robust_regions["bootstrap_frequency_min"].isna().all()
print("Quality axis partitioned into", len(thresholds), "atomic regions")


## Run 13 - Publication figures

Ideal points use hollow or hatched styling. Projected energy is shown only as a
break-even boundary, never as a measured or numerically identified point.


In [ ]:
FIGURE_NAMES = []
plt.style.use("seaborn-v0_8-whitegrid")

def save_figure(stem):
    for suffix in ["png", "pdf"]:
        name = f"fig_{stem}.{suffix}"
        path = output_path(name)
        path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(path, dpi=220 if suffix == "png" else None, bbox_inches="tight")
        FIGURE_NAMES.append(name)
    plt.close()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex="col")
for col, model in enumerate(["llama", "qwen"]):
    variants = ["C0", "C1", "C2", "C3", "C4"] if model == "llama" else ["Q0", "Q1-W", "Q2", "Q3-Ref", "Q4-Ref"]
    for variant in variants:
        group = energy.query("model == @model and source_variant == @variant and source_table == 'core' and regime == 'raw'").groupby("context").agg(energy=("total_energy_j", "mean"), throughput=("decode_tok_s", "mean")).reset_index()
        axes[0, col].plot(group.context, group.energy, marker="o", label=variant)
        axes[1, col].plot(group.context, group.throughput, marker="o", label=variant)
    axes[0, col].set_title(model.title() + " raw energy")
    axes[1, col].set_title(model.title() + " measured throughput")
    axes[0, col].set_ylabel("J")
    axes[1, col].set_ylabel("decode tok/s")
    axes[1, col].set_xlabel("context tokens")
    axes[0, col].legend(fontsize=8)
fig.suptitle("Within-model context scaling; Llama C1 and Qwen Q1-W baselines differ")
save_figure("energy_throughput_vs_context")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, model in zip(axes[:2], ["llama", "qwen"]):
    table = quality_intervals.query("model == @model and quality_population == 'hard60' and depth == 'all'").copy()
    x = np.arange(len(table))
    ax.errorbar(x, table.estimate, yerr=[table.estimate-table.wilson95_low, table.wilson95_high-table.estimate], fmt="o", capsize=4)
    ax.set_xticks(x, table.source_variant, rotation=35, ha="right")
    ax.set_ylim(0, 1.02); ax.set_ylabel("hard retrieval rate")
    ax.set_title(model.title() + " hard60 Wilson 95%")
common_plot = quality_intervals.query("model == 'qwen' and quality_population == 'qwen_common15' and depth == 'all'").copy()
x = np.arange(len(common_plot))
axes[2].errorbar(x, common_plot.estimate,
    yerr=[common_plot.estimate-common_plot.wilson95_low, common_plot.wilson95_high-common_plot.estimate], fmt="o", capsize=4)
axes[2].set_xticks(x, common_plot.source_variant, rotation=35, ha="right")
axes[2].set_ylim(0, 1.02); axes[2].set_ylabel("retrieval rate")
axes[2].set_title("Qwen common-15 TQ 2/3/4-bit")
fig.suptitle("Tokenizer-specific 4096-token materializations; no architecture-only causal claim")
save_figure("hard_quality_wilson")

plot_interactions = interactions.query("metric == 'total_energy_j'")
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, model in enumerate(["llama", "qwen"]):
    subset = plot_interactions.query("model == @model")
    ax.plot(subset.regime, subset.mean_interaction, marker="o", label=model)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("difference-in-differences (J)")
ax.set_title("Sequential compound interaction; capped values are trace reintegrations")
ax.legend()
save_figure("interaction_by_power_regime")

subset = storage_comparison.query("phase == 'decode' and nominal_tq_bits == 4")
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(subset)); width = .25
ax.bar(x-width, subset.measured_non_tq_live_bytes/2**20, width, label="measured non-TQ live")
ax.bar(x, subset.realized_ref_live_bytes/2**20, width, label="realized TQ-Ref live")
ax.bar(x+width, subset.ideal_aligned_bytes/2**20, width, facecolor="none", edgecolor="black", hatch="//", label="ideal packed persistent")
ax.set_xticks(x, [f"{m}\n{v}" for m, v in zip(subset.model, subset.source_variant)])
ax.set_ylabel("persistent/live cache MiB")
ax.set_title("Measured versus realized-reference versus ideal persistent storage")
ax.legend()
save_figure("measured_realized_ideal_storage")

subset = break_even.query("phase == 'total' and regime == 'raw'")
fig, ax = plt.subplots(figsize=(10, 5))
ordered = subset.sort_values(["model", "projected_variant"])
ax.barh(np.arange(len(ordered)), ordered.delta_d_bytes/1e9, facecolor="none", edgecolor="black", hatch="//")
ax.set_yticks(np.arange(len(ordered)), [f"{m}: {v}" for m, v in zip(ordered.model, ordered.projected_variant)])
ax.set_xlabel("Delta_D total (GB of modeled logical demand)")
ax.set_title("Break-even when O_net / epsilon_mem < Delta_D, for epsilon_mem > 0")
save_figure("energy_break_even_regions")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for model, marker in [("llama", "o"), ("qwen", "s")]:
    table = empirical_pareto.query("model == @model")
    axes[0].scatter(table.energy_mean_j, table.retrieval_estimate, marker=marker, label=model)
    frontier = table.query("point_frontier_member").sort_values("energy_mean_j")
    axes[0].plot(frontier.energy_mean_j, frontier.retrieval_estimate, linestyle="--", alpha=.7)
    for row in table.itertuples():
        axes[0].annotate(row.source_variant, (row.energy_mean_j, row.retrieval_estimate), fontsize=7)
axes[0].set_xlabel("measured raw energy (J)"); axes[0].set_ylabel("hard retrieval")
axes[0].set_title("Empirical measured frontier inputs")
axes[0].legend()

robust_view = robust_regions.query("quality_population == 'hard60' and feasibility_rule.str.startswith('conservative') and quality_realization_scenario == 'strict_output_equivalence' and realization_scenario == 'zero_net_overhead_symbolic'").copy()
breakpoints = sorted(set([0.0, 1.0] + list(projection_quality["wilson_low"]) + list(projection_quality["wilson_high"])))
region_order = [item[0] for item in thresholds]
point_order = sorted(robust_view.point_id.unique())
class_code = {"none_quality_excluded": 0, "none_universally_dominated": 0,
              "some_conditional_region": 1, "all_by_strict_memory_minimum": 2}
matrix = np.full((len(point_order), len(region_order)), np.nan)
for i, point_id in enumerate(point_order):
    lookup = robust_view.query("point_id == @point_id").set_index("a_min_region_id").robust_membership_class
    for j, region_id in enumerate(region_order):
        if region_id in lookup.index:
            matrix[i, j] = class_code[lookup.loc[region_id]]
        else:
            matrix[i, j] = 0

image = axes[1].pcolormesh(np.asarray(breakpoints), np.arange(len(point_order)+1), matrix,
    vmin=0, vmax=2, shading="flat",
    cmap=matplotlib.colors.ListedColormap(["#b84a4a", "#d8a72d", "#2c8c68"]))
axes[1].set_yticks(np.arange(len(point_order)) + .5, point_order, fontsize=7)
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("A_min quality threshold (proportional axis)")
axes[1].set_title("Conservative zero-overhead robust frontier class")
colorbar = fig.colorbar(image, ax=axes[1], ticks=[0, 1, 2])
colorbar.ax.set_yticklabels(["none", "some", "all"])
save_figure("empirical_and_projected_pareto")

print("Generated", len(FIGURE_NAMES), "figure files")


## Run 14 - Machine-readable outputs and findings reports

All tables, assumptions, source hashes, protocol formulas, and claim boundaries
are emitted before the output manifest and final-integrity marker.


In [ ]:
assert OUTPUT_ROOT.is_dir()
assert sorted(path.name for path in OUTPUT_ROOT.iterdir() if path.is_file()) == sorted(FIGURE_NAMES)
assert not any(path.is_dir() for path in OUTPUT_ROOT.iterdir())

tables = {
    "source_validation.csv": pd.DataFrame(SOURCE_GATES),
    "normalized_energy_rows.csv": energy,
    "normalized_quality_rows.csv": quality,
    "within_model_effects.csv": within_effects,
    "cross_model_interaction_summary.csv": interactions,
    "quality_wilson_intervals.csv": quality_intervals,
    "power_regime_stability.csv": power_stability,
    "registered_significance_results.csv": registered_significance,
    "registered_inferential_tests.csv": registered_inferential_tests,
    "cross_model_quality_contrasts.csv": cross_model_quality_contrasts,
    "empirical_pareto.csv": empirical_pareto,
    "storage_calibration_tests.csv": storage_calibration,
    "counterfactual_eligibility.csv": eligibility,
    "measured_realized_ideal_storage.csv": storage_comparison,
    "ideal_packed_kv_tensor_accounting.csv": tensor_accounting,
    "ideal_packed_kv_footprint.csv": footprints,
    "modeled_phase_byte_demand.csv": demand,
    "energy_break_even_surface.csv": break_even,
    "quality_constraint_regions.csv": quality_regions,
    "conditional_ideal_pareto.csv": conditional_pareto,
    "robust_optimality_regions.csv": robust_regions,
}
for name, frame in tables.items():
    write_csv(name, frame)

write_json("ideal_tq_assumptions.json", assumptions)
write_json("source_validation.json", {"schema_version": SCHEMA, "gates": SOURCE_GATES,
    "all_passed": all(row["passed"] for row in SOURCE_GATES)})

input_manifest = {"schema_version": SCHEMA, "project_root_name": PROJECT_ROOT.name,
    "notebook_source_sha256": NOTEBOOK_SOURCE_SHA256,
    "sources": [{"source_id": key, "root": value["root"].name,
                 "schema": value["schema"], "fingerprint": value["fingerprint"]}
                for key, value in EXPECTED.items()],
    "files": sorted(INPUT_FILES, key=lambda x: (x["source_id"], x["path"]))}
write_json("input_manifest.json", input_manifest)

protocol = {"schema_version": SCHEMA, "notebook_source_sha256": NOTEBOOK_SOURCE_SHA256,
    "runtime": {"python": platform.python_version(), "numpy": np.__version__,
                "pandas": pd.__version__, "matplotlib": matplotlib.__version__},
    "bootstrap": {"repetitions": BOOTSTRAP_REPS, "seed": BOOTSTRAP_SEED,
        "energy_quality_resampled_independently": True,
        "frequency_interpretation": "resampling selection stability; not truth probability"},
    "formulas": {"fp_kv": "2*H*d*s_fp*sum_l(B_l)",
        "ideal_kv": "2*H*sum_l((B-r)*(d*q/8+s_norm)+r*d*s_fp), tensorwise ceiling/alignment",
        "energy": "E_ideal=E_baseline-epsilon_mem*Delta_D+O_net",
        "break_even": "O_net<epsilon_mem*Delta_D"},
    "baseline_policy": {"llama_context_matrix": "stock C1", "llama_4096_control": "separate-session C1-W descriptive",
        "qwen_context_matrix": "matched-wrapper Q1-W", "compound_baseline": "C2/Q2"},
    "quality_policy": {"hard": "60 tokenizer-specific materializations per model",
        "qwen_low_bits": "common 15 scenario IDs", "llama_low_bits": "excluded from hard Pareto",
        "strict_output_equivalence_required": True,
        "delta_impl_domain": "-wilson_low <= delta_impl <= 1-wilson_high"},
    "parameter_domains": {"epsilon_mem": None, "O_net": None},
    "raw_joules_primary": True, "capped_values_are_trace_sensitivities": True}
write_json("analysis_protocol.json", protocol)

llama_q = empirical_pareto.query("model == 'llama'").set_index("source_variant")
qwen_q = empirical_pareto.query("model == 'qwen'").set_index("source_variant")
findings = f"""# Cross-Model Synthesis Findings

## Evidence boundary

- All accepted source manifests, identities, row counts, CSV correspondences, and 195 traces passed before analysis.
- Raw joules are primary. The 82 W and 70 W values are clipped-trace sensitivity integrations, not hardware reruns.
- Llama systems effects use historical stock C1; Qwen uses matched-wrapper Q1-W. Llama's C1-W control is a separate-session 4096-only descriptive comparison.
- Each model uses the same semantic scenarios but tokenizer-specific exact 4096-token materialization. Architecture alone is not identified as the cause of cross-model differences.

## Empirical results

- Llama hard retrieval: C1-W {int(llama_q.loc['C1-W','retrieval_successes'])}/60, C2 {int(llama_q.loc['C2','retrieval_successes'])}/60, C3-Ref {int(llama_q.loc['C3','retrieval_successes'])}/60, C4-Ref {int(llama_q.loc['C4','retrieval_successes'])}/60.
- Qwen hard retrieval: Q1-W {int(qwen_q.loc['Q1-W','retrieval_successes'])}/60, Q2 {int(qwen_q.loc['Q2','retrieval_successes'])}/60, Q3-Ref {int(qwen_q.loc['Q3-Ref','retrieval_successes'])}/60, Q4-Ref {int(qwen_q.loc['Q4-Ref','retrieval_successes'])}/60.
- Qwen eviction collapsed hard retrieval while Q3-Ref retained Q1-W-like retrieval at substantially higher measured energy. The Llama hard run did not show the same eviction collapse.
- Sequential compound interactions and rankings retained their registered raw/capped interpretation. Five blocks limit Wilcoxon resolution.

## Provenance caveat

Notebook 08 required an in-session quality-validator correction and capped-analysis merge correction. The exported notebook preserves both fixes and persisted artifacts are hash-consistent, but fingerprint `861d89617e53034d` and its artifact manifest do not cryptographically bind repair-cell source bytes. This is a reproducibility caveat, not a measurement-integrity failure.

## Ideal realization gap

- FP retained-byte and uint8 logical-payload calibration gates passed for both models and TQ-only/compound paths.
- Under the declared perfectly packed persistent layout, all projected 2/3/4-bit rows are smaller than matched FP16 persistent KV because packed-index savings exceed norms, residuals, alignment, and metadata.
- This proves only conditional persistent-KV memory dominance under the declared representation. It does not prove total peak-VRAM, latency, hardware traffic, energy, or power improvement.
- Ideal energy is only identified by `E_ideal = E_baseline - epsilon_mem * Delta_D + O_net`. No numerical parameter domain was justified, so no numerical ideal-energy or projected energy-Pareto point was produced.
- Conservative, point, and possible quality rules are exported separately across the complete threshold partition. Unobserved 2/3-bit compound rows remain storage-only.
"""
write_text("synthesis_findings.md", findings)

realization = """# Ideal-TQ Realization-Gap Summary

The measured TQ-Ref path is numerically active, uses one uint8 index per coordinate at nominal 2/3/4 bits, retains materialized K/V, and is unfused. It is not a simulation and not a physically packed cache.

The analytical ideal removes persistent materialized K/V, packs indices at the observed nominal bit width, retains FP32 norms and a 128-token FP16 residual, and assumes reconstruction remains in registers/shared memory. Tensor descriptors are reconstructed from runtime-asserted contracts and reconciled to persisted aggregate bytes; no direct live tensor dump was preserved.

Unit gates reproduced retained FP16 bytes and eight-bit package payloads exactly before projection. Bit-width and budget monotonicity passed. Persistent-memory dominance is proved only where index savings exceed norm, residual, alignment, and metadata overhead.

Prefill and decode modeled byte demand are separate. The model is logical compulsory algorithmic demand, not exact hardware DRAM traffic. Total break-even uses both phases. Because no justified finite domains exist for `epsilon_mem` or `O_net`, break-even remains symbolic and robust bootstrap-frequency bounds remain unavailable. This fail-closed result prevents fabricated ideal joules.

Quality transfers only under strict output equivalence. Four-bit TQ-only and compound hard paths are eligible; Qwen low-bit TQ-only comparisons use the common 15 scenarios; Llama historical low-bit quality and all unobserved low-bit compounds are excluded from hard-quality energy Pareto claims.
"""
write_text("realization_gap_summary.md", realization)
print("Wrote tables, manifests, and findings")


## Run 15 - Output manifest and final integrity

The final marker is written last after exact inventory, proof, evidence-tier, and
non-identification assertions. Any mismatch aborts without an acceptance marker.


In [ ]:
EXPECTED_OUTPUTS_BEFORE_MANIFEST = sorted(list(tables) + [
    "ideal_tq_assumptions.json", "source_validation.json", "input_manifest.json",
    "analysis_protocol.json", "synthesis_findings.md", "realization_gap_summary.md",
] + FIGURE_NAMES)
observed_before = sorted(str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/")
                         for path in OUTPUT_ROOT.rglob("*") if path.is_file())
assert observed_before == EXPECTED_OUTPUTS_BEFORE_MANIFEST, (set(observed_before) ^ set(EXPECTED_OUTPUTS_BEFORE_MANIFEST))

output_manifest = {"schema_version": SCHEMA, "notebook_source_sha256": NOTEBOOK_SOURCE_SHA256,
    "files": [{"path": name, "bytes": output_path(name).stat().st_size,
               "sha256": sha256_file(output_path(name))}
              for name in EXPECTED_OUTPUTS_BEFORE_MANIFEST]}
write_json("output_manifest.json", output_manifest)

fingerprint_payload = {"schema": SCHEMA, "notebook_source_sha256": NOTEBOOK_SOURCE_SHA256,
    "input_manifest_sha256": sha256_file(output_path("input_manifest.json")),
    "analysis_protocol_sha256": sha256_file(output_path("analysis_protocol.json")),
    "ideal_tq_assumptions_sha256": sha256_file(output_path("ideal_tq_assumptions.json"))}
ANALYSIS_FINGERPRINT = canonical_sha256(fingerprint_payload)[:16]

proof_gates = {
    "fp_storage_reproduction": bool(storage_calibration.passed.all()),
    "aggregate_uint8_payload_reproduction": bool(storage_calibration.passed.all()),
    "persisted_category_bytes_reconciled_where_available": bool(storage_calibration.query("norm_category_directly_persisted").passed.all()),
    "all_llama_hard_category_rows_and_states_reconciled": bool(hard_category_rows_validated == 120 and len(storage_calibration[storage_calibration.phase.str.startswith("hard_generation_final")]) == 4),
    "bit_monotonicity": True, "budget_monotonicity": True,
    "memory_dominance_conditions_verified": bool(footprints.persistent_memory_dominance_proved.all()),
    "phase_models_passed": bool(demand.phase_model_passed.all()),
    "all_divided_break_even_rows_have_positive_delta": bool(break_even.query("break_even_available").delta_d_bytes.gt(0).all()),
    "nonpositive_delta_rows_have_no_division": bool(break_even.query("not break_even_available").break_even_epsilon_expression.isna().all()),
    "no_unobserved_path_received_quality_or_energy": not bool(eligibility.query("not observed_numerical_path and quality_energy_pareto_eligible").shape[0]),
    "no_numerical_epsilon_or_o_net_domain": not bool(break_even.numerical_parameter_domain_declared.any()),
    "no_numerical_ideal_energy_points": bool(break_even.numerical_ideal_energy_j.isna().all()),
    "quality_axis_exactly_partitioned": thresholds[0][1] == 0.0 and thresholds[-1][2] == 1.0,
    "qwen_common15_has_2_3_4_bit_paths": bool(set(projection_quality.query("quality_population == 'qwen_common15'").nominal_tq_bits) == {2, 3, 4}),
    "symbolic_pairwise_dominance_emitted": bool(conditional_pareto.pairwise_dominance_conditions_json.map(json.loads).map(lambda x: isinstance(x, list)).all()),
    "no_empty_symbolic_membership_expression": bool(~conditional_pareto.pareto_membership_condition.str.contains("NOT\\(\\)", regex=True).any()),
    "quality_change_parameter_propagated_to_pareto": bool(conditional_pareto.quality_admission_condition.str.contains("delta_impl").any()),
    "quality_change_parameter_domain_bounded_to_valid_rate": bool((conditional_pareto.dropna(subset=["quality_delta_lower_bound"])["quality_delta_lower_bound"] >= -1).all() and (conditional_pareto.dropna(subset=["quality_delta_upper_bound"])["quality_delta_upper_bound"] <= 1).all() and (conditional_pareto.dropna(subset=["quality_delta_lower_bound"])["quality_delta_lower_bound"] <= conditional_pareto.dropna(subset=["quality_delta_upper_bound"])["quality_delta_upper_bound"]).all() and assumptions["quality_model"]["delta_impl_domain"].startswith("-wilson_low")),
    "zero_overhead_universal_dominance_simplified": bool((robust_regions.query("point_id == 'measured_llama_C2' and realization_scenario == 'zero_net_overhead_symbolic' and quality_realization_scenario == 'strict_output_equivalence' and a_min_region_id == 'point-00'").robust_membership_class == "none_universally_dominated").all()),
    "adjusted_p_values_attached_where_registered": bool(within_effects[(within_effects.context == 4096) & within_effects.design.str.startswith("paired_within")].p_adjusted.notna().all()),
    "empirical_and_projected_evidence_separated": bool(set(empirical_pareto.evidence_tier) == {"measured_system"} and {"measured_system", "realized_reference", "ideal_counterfactual"}.issubset(set(conditional_pareto.evidence_tier))),
}
for source_file in INPUT_FILES:
    source_path = PROJECT_ROOT / source_file["path"]
    assert source_path.stat().st_size == source_file["bytes"]
    assert sha256_file(source_path) == source_file["sha256"]
proof_gates["sources_unchanged_after_analysis"] = True
assert all(proof_gates.values()), proof_gates

expected_final_inventory = sorted(EXPECTED_OUTPUTS_BEFORE_MANIFEST + ["output_manifest.json", "final_integrity.json"])
integrity = {"passed": True, "schema_version": SCHEMA, "fingerprint": ANALYSIS_FINGERPRINT,
    "notebook_source_sha256": NOTEBOOK_SOURCE_SHA256,
    "input_manifest_sha256": fingerprint_payload["input_manifest_sha256"],
    "analysis_protocol_sha256": fingerprint_payload["analysis_protocol_sha256"],
    "ideal_tq_assumptions_sha256": fingerprint_payload["ideal_tq_assumptions_sha256"],
    "output_manifest_sha256": sha256_file(output_path("output_manifest.json")),
    "source_gates": {"count": len(SOURCE_GATES), "all_passed": True},
    "row_counts": {name: len(frame) for name, frame in tables.items()},
    "evidence_limitations": {"direct_live_tensor_dump_available": False,
        "shape_dtype_validation_scope": "runtime-asserted contracts plus persisted aggregate/category bytes; realized residual backing storage is separate from logical residual payload"},
    "proof_gates": proof_gates, "expected_outputs": expected_final_inventory,
    "unexpected_outputs": [], "completed_at_utc": datetime.now(timezone.utc).isoformat()}
write_json("final_integrity.json", integrity)

observed_final = sorted(str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/")
                        for path in OUTPUT_ROOT.rglob("*") if path.is_file())
assert observed_final == expected_final_inventory
assert read_json(output_path("final_integrity.json"))["passed"] is True
print(json.dumps({"passed": True, "fingerprint": ANALYSIS_FINGERPRINT,
                  "output_files": len(observed_final), "output_root": str(OUTPUT_ROOT)}, indent=2))


## Handoff

Use `final_integrity.json` as the synthesis acceptance marker only after this
notebook completes without an exception. The Markdown reports are concise paper
inputs; canonical values remain in the emitted CSV/JSON products and accepted
source bundles. Do not modify accepted Llama or Qwen evidence.
